# CPSC Recall Change Monitoring System

This notebook is the Colab version

The actual workflow implemented in the code is:

1. Mount Google Drive and configure snapshot, cache, output, LLM, retrieval, and human-gold paths.
2. Fetch or reuse the current CPSC recall snapshot.
3. Load the explicit previous snapshot and the current snapshot.
4. Detect added, removed, and modified recall change units.
5. Compute importance scores, route labels, route priorities, severity signals, numeric deltas, and actionability scores.
6. Build a recall-level retrieval corpus and a TF-IDF based hybrid retriever.
7. Run four generation methods: Prompt-Only, RAG, Agentic RAG, and Contrastive Adaptive RAG.
8. Apply the later patch cells to improve retrieval coverage, evidence coverage repair, contrastive verification, coverage-first contrastive generation, and precision cleanup.
9. Evaluate the generated summaries using frozen human gold when available, otherwise proxy gold.
10. Export summaries, automatic evaluation results, route digests, human evaluation templates, demo notes, and the best final summary text file.

For debugging, use `LLM_MODE="offline"` to verify paths and data flow without spending tokens. For the final experiment, switch to `LLM_MODE="openai_compatible"` and provide a valid API key at runtime.

## Required Files and Runtime Settings

This section should match the paths and variables used in the current code.

### Required input file

1. **Previous snapshot baseline CSV**
   - Code variable: `EXPLICIT_PREVIOUS_SNAPSHOT`
   - Current default path in the notebook: `/content/drive/MyDrive/GR5293 HW/final project/try/snapshots/cpsc_recalls_20260304.csv`
   - Purpose: this file is the historical baseline used to compare against the current CPSC snapshot.

### Automatically downloaded or reused file

2. **Current snapshot CSV**
   - Code function: `fetch_current_snapshot(SNAPSHOT_DIR)`
   - The notebook first checks whether today's file already exists under `SNAPSHOT_DIR`.
   - If it exists, the cached snapshot is reused.
   - If it does not exist, the notebook downloads the CPSC CSV file first and falls back to the CPSC Recall API if the CSV request fails.
   - Output filename pattern: `cpsc_recalls_YYYYMMDD.csv`.

### Recommended evaluation file

3. **Frozen human gold CSV**
   - Code variable: `HUMAN_GOLD_PATH`
   - Current default path in the notebook: `/content/drive/MyDrive/GR5293 HW/final project/try/snapshots/frozen_human_gold_current_window_v2.csv`
   - Purpose: provides reference recall numbers and change units for automatic evaluation.
   - If this file is missing, the notebook can still run with proxy gold, but proxy gold should only be used for debugging.

### Runtime setting

4. **OpenAI-compatible API key**
   - Required only when `LLM_MODE="openai_compatible"`.
   - For safety, the key should be supplied through an environment variable or entered with `getpass` at runtime, not hard-coded in the notebook.
   - For a no-token dry run, set `LLM_MODE="offline"` before running generation.

## Original File Notes

Its header records these design changes: Contrastive Adaptive RAG, raw-versus-final summary comparison, three-tier claim grounding, retrieval ground-truth decoupling from `importance_score`, route-aware evaluation, and IAA placeholder columns for human evaluation templates.


In [ ]:
# -*- coding: utf-8 -*-
import os, re, json, math, shutil, requests
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 0) Configuration

This cell mounts Google Drive when running in Colab, defines `ROOT_DIR`, `SNAPSHOT_DIR`, `OUTPUT_DIR`, and `CACHE_DIR`, sets the CPSC CSV/API source URLs, configures LLM mode and OpenAI-compatible settings, points to the frozen human-gold file, defines retrieval budgets, and sets the explicit previous snapshot path.

It also defines the core comparison columns, severe-term weights, field bonuses, route order, and the global recall-candidate cache used later by retrieval.

In [ ]:
# 0) CONFIG


def mount_google_drive_if_possible():
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
    except Exception as e:
        print(f"Google Drive mount skipped: {e}")

mount_google_drive_if_possible()

ROOT_DIR      = "/content/drive/MyDrive/GR5293 HW/final project/try"
SNAPSHOT_DIR  = os.path.join(ROOT_DIR, "snapshots")
OUTPUT_DIR    = SNAPSHOT_DIR
CACHE_DIR     = os.path.join(ROOT_DIR, "cache")

for _d in [ROOT_DIR, SNAPSHOT_DIR, OUTPUT_DIR, CACHE_DIR]:
    os.makedirs(_d, exist_ok=True)

CPSC_CSV_URL       = "https://www.cpsc.gov/s3fs-public/recall-data/recalls_recall_listing.csv"
CPSC_RECALL_API_URL = "https://www.saferproducts.gov/RestWebServices/Recall?format=json"

# Generation mode: "offline" | "openai_compatible"
LLM_MODE        = "openai_compatible"
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")

# For final LLM experiments keep LLM_MODE="openai_compatible".
# For a quick no-token dry run set LLM_MODE="offline" before running generation.
if LLM_MODE == "openai_compatible" and not OPENAI_API_KEY:
    try:
        from getpass import getpass
        OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (leave empty only if using offline mode): ").strip()
        if OPENAI_API_KEY:
            os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    except Exception as e:
        print(f"API key prompt skipped: {e}")
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_MODEL = "gpt-4o-mini"
LLM_TEMPERATURE = 0.2

HUMAN_GOLD_PATH  = "/content/drive/MyDrive/GR5293 HW/final project/try/snapshots/frozen_human_gold_current_window_v2.csv"
USE_PROXY_DEBUG  = True

RAG_MAX_DOCS   = 10
AGENT_MAX_DOCS = 10
RETRIEVAL_KS   = [5, 10]

EXPLICIT_PREVIOUS_SNAPSHOT = os.path.join(SNAPSHOT_DIR, "cpsc_recalls_20260304.csv")

KEY_COL = "Recall Number"
COMPARE_COLS = [
    "Recall Heading", "Name of product", "Description",
    "Hazard Description", "Consumer Action", "Remedy Type",
    "Units", "Incidents", "Remedy",
    "Importers", "Manufacturers", "Distributors", "Manufactured In",
]

RECALL_CANDIDATE_CACHE: Dict[Any, pd.DataFrame] = {}

SEVERE_TERM_WEIGHTS = {
    "death": 8.0, "fatal": 8.0, "injury": 4.0, "injuries": 4.0,
    "burn": 4.0, "fire": 4.0, "battery": 3.0, "lithium": 3.0,
    "children": 3.0, "child": 3.0, "infant": 3.0,
    "choking": 5.0, "suffocation": 6.0, "strangulation": 6.0,
    "fall": 3.0, "drowning": 5.0, "shock": 4.0, "electrocution": 8.0,
    "overheat": 3.0, "overheating": 3.0, "explosion": 6.0,
    "carbon monoxide": 7.0, "poison": 6.0,
    "stop using": 2.5, "stop use": 2.5, "refund": 2.0,
    "repair": 1.5, "flammability": 3.0,
}

KEY_FIELD_BONUS = {
    "Consumer Action": 4.0, "Remedy": 3.5,
    "Hazard Description": 3.0, "Incidents": 2.5,
    "Units": 2.0, "Remedy Type": 2.0,
}

ROUTE_ORDER = {
    "new_high_risk": 0,
    "consumer_action_update": 1,
    "incident_escalation": 2,
    "removed_or_resolved": 3,
    "metadata_or_context_update": 4,
}


## 1) UTILS

General tool functions: text cleaning, recall number normalization, sentence segmentation, JSON writing, HTTP headers, LLM calls, etc.

In [ ]:
# 1) UTILS


def slug_date(ts: Optional[pd.Timestamp] = None) -> str:
    return (ts or pd.Timestamp.utcnow().normalize()).strftime("%Y%m%d")

def normalize_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).replace("\xa0", " ")).strip()

soft_str = normalize_text

def ensure_columns(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df

def sent_split(text: str) -> List[str]:
    text = normalize_text(text)
    return [p.strip() for p in re.split(r"(?<=[\.!?])\s+", text) if p.strip()] if text else []

def safe_int_from_text(x: str) -> Optional[int]:
    nums = re.findall(r"\d[\d,]*", str(x))
    try:
        return int(nums[0].replace(",", "")) if nums else None
    except Exception:
        return None

def canon_recall_number(x: Any) -> str:
    raw = normalize_text(x)
    if not raw:
        return ""
    m = re.search(r"\b(\d{2})-(\d{3,4})\b", raw)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    digits = re.sub(r"\D", "", raw)
    if not digits:
        return ""
    if len(digits) == 5:
        return f"{digits[:2]}-{digits[2:]}"
    if len(digits) == 4:
        return f"0{digits[0]}-{digits[1:]}"
    if 6 <= len(digits) <= 8:
        return f"{digits[:2]}-{digits[2:]}"
    return raw

def extract_recall_numbers(text: str) -> List[str]:
    raw = re.findall(r"\b\d{2}-\d{3,4}\b|\b\d{4,8}\b", normalize_text(text))
    return list(dict.fromkeys(filter(None, map(canon_recall_number, raw))))

def now_ts() -> str:
    return pd.Timestamp.utcnow().isoformat()

def write_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def browser_headers() -> Dict[str, str]:
    return {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
        ),
        "Accept": "text/csv,application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.cpsc.gov/Recalls",
        "Cache-Control": "no-cache",
    }

def call_llm(messages: List[Dict[str, str]],
             model: str = OPENAI_MODEL,
             temperature: float = LLM_TEMPERATURE) -> Tuple[str, Dict[str, Any]]:
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is empty but LLM_MODE='openai_compatible'.")
    url = OPENAI_BASE_URL.rstrip("/") + "/chat/completions"
    resp = requests.post(
        url,
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}",
                 "Content-Type": "application/json"},
        json={"model": model, "messages": messages, "temperature": temperature},
        timeout=180,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"], data.get("usage", {})

def _num(x, default=0.0) -> float:
    try:
        v = float(x)
        return default if math.isnan(v) else v
    except Exception:
        return default

def parse_snapshot_date_from_name(path: str) -> pd.Timestamp:
    m = re.match(r"^cpsc_recalls_(\d{8}|\d{4}_\d{2}_\d{2})\.csv$", os.path.basename(path))
    if not m:
        return pd.Timestamp("1970-01-01")
    raw = m.group(1)
    fmt = "%Y_%m_%d" if "_" in raw else "%Y%m%d"
    try:
        return pd.to_datetime(raw, format=fmt)
    except Exception:
        return pd.Timestamp("1970-01-01")


## 2) DATA INGESTION

Data reading and retrieval: Retrieve the current snapshot from CPSC CSV/API, read the previous/current snapshot, and check if the baseline file exists.

In [ ]:
# 2) DATA INGESTION

def _join_vals(items, key) -> str:
    vals = []
    if isinstance(items, list):
        for x in items:
            if isinstance(x, dict):
                v = normalize_text(x.get(key, ""))
                if v:
                    vals.append(v)
    return " | ".join(sorted(set(vals)))

def api_record_to_row(rec: Dict[str, Any]) -> Dict[str, Any]:
    products = rec.get("Products", [])
    hazards  = rec.get("Hazards", [])
    remedies = rec.get("Remedies", [])
    return {
        KEY_COL:              normalize_text(rec.get("RecallNumber", "")),
        "Recall Heading":     normalize_text(rec.get("Headline", "")),
        "Name of product":    _join_vals(products, "Name"),
        "Description":        _join_vals(products, "Description"),
        "Hazard Description": _join_vals(hazards, "Name"),
        "Consumer Action":    normalize_text(rec.get("ConsumerContact", "")),
        "Remedy Type":        _join_vals(remedies, "Name"),
        "Units":              normalize_text(str(rec.get("NumberOfUnits", ""))),
        "Incidents":          normalize_text(rec.get("Inconjunctions", "")),
        "Remedy":             _join_vals(remedies, "Name"),
        "Importers":          _join_vals(rec.get("Importers", []), "Name"),
        "Manufacturers":      _join_vals(rec.get("Manufacturers", []), "Name"),
        "Distributors":       _join_vals(rec.get("Retailers", []), "Name"),
        "Manufactured In":    _join_vals(rec.get("ManufacturerCountries", []), "Country"),
    }

def fetch_via_csv(today_file: str, csv_url: str = CPSC_CSV_URL) -> str:
    resp = requests.get(csv_url, headers=browser_headers(), timeout=120)
    resp.raise_for_status()
    with open(today_file, "wb") as f:
        f.write(resp.content)
    return today_file

def fetch_via_api(today_file: str, api_url: str = CPSC_RECALL_API_URL) -> str:
    rows, limit, offset = [], 100, 0
    while True:
        resp = requests.get(
            api_url, params={"limit": limit, "offset": offset},
            headers=browser_headers(), timeout=60
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        rows.extend([api_record_to_row(r) for r in batch])
        if len(batch) < limit:
            break
        offset += limit
    pd.DataFrame(rows).to_csv(today_file, index=False)
    return today_file

def fetch_current_snapshot(snapshot_dir: str = SNAPSHOT_DIR) -> str:
    today_file = os.path.join(snapshot_dir, f"cpsc_recalls_{slug_date()}.csv")
    if os.path.exists(today_file):
        print(f"Using cached snapshot: {today_file}")
        return today_file
    print("Fetching current snapshot via CSV …")
    try:
        return fetch_via_csv(today_file)
    except Exception as e:
        print(f"CSV fetch failed ({e}), trying API …")
        return fetch_via_api(today_file)

def load_snapshot(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str, low_memory=False)
    df = ensure_columns(df, [KEY_COL] + COMPARE_COLS)
    for c in df.columns:
        df[c] = df[c].map(normalize_text)
    df = df[df[KEY_COL].map(canon_recall_number) != ""].copy()
    df[KEY_COL] = df[KEY_COL].map(lambda x: canon_recall_number(x) or normalize_text(x))
    return df.drop_duplicates(subset=[KEY_COL], keep="last").reset_index(drop=True)

def list_snapshot_files(snapshot_dir: str = SNAPSHOT_DIR) -> List[str]:
    files = [
        os.path.join(snapshot_dir, f)
        for f in os.listdir(snapshot_dir)
        if re.match(r"cpsc_recalls_\d{8}\.csv", f)
    ]
    return sorted(files, key=parse_snapshot_date_from_name)

def check_required_previous_snapshot(path: str = EXPLICIT_PREVIOUS_SNAPSHOT):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Previous snapshot not found: {path}\n"
            "Place the baseline CSV there before running."
        )
    print(f"Previous snapshot OK: {path}")


## 3) CHANGE DETECTION

Change detection: Compare the previous snapshot with the current snapshot, generate change units including added, removed, and modified, and calculate importance_store.

In [ ]:
# 3) CHANGE DETECTION


def compute_importance(row: Dict[str, Any]) -> float:
    text = " ".join([
        soft_str(row.get("Hazard Description", "")),
        soft_str(row.get("Consumer Action", "")),
        soft_str(row.get("Remedy", "")),
        soft_str(row.get("Incidents", "")),
        soft_str(row.get("Recall Heading", "")),
    ]).lower()
    score = sum(w for term, w in SEVERE_TERM_WEIGHTS.items() if term in text)
    score += KEY_FIELD_BONUS.get(normalize_text(row.get("changed_field", "")), 0.0)
    change_type = normalize_text(row.get("change_type", "")).lower()
    if change_type == "added":
        score += 3.0
    elif change_type == "modified":
        score += 1.5
    return round(score, 3)

def build_change_units(prev_df: pd.DataFrame, cur_df: pd.DataFrame) -> pd.DataFrame:
    prev_map = {r[KEY_COL]: r.to_dict() for _, r in prev_df.iterrows()}
    cur_map  = {r[KEY_COL]: r.to_dict() for _, r in cur_df.iterrows()}

    rows = []
    uid  = 0

    # Added
    for rn in sorted(set(cur_map) - set(prev_map)):
        rec = cur_map[rn]
        rec_text = " ".join(soft_str(rec.get(c, "")) for c in COMPARE_COLS).lower()
        score = sum(w for t, w in SEVERE_TERM_WEIGHTS.items() if t in rec_text) + 3.0
        row = {
            "change_unit_id": f"cu_{uid:05d}", KEY_COL: rn,
            "change_type": "added", "changed_field": "",
            "old_value": "", "new_value": "",
            "importance_score": round(score, 3),
        }
        row.update({c: soft_str(rec.get(c, "")) for c in COMPARE_COLS})
        rows.append(row); uid += 1

    # Removed
    for rn in sorted(set(prev_map) - set(cur_map)):
        rec = prev_map[rn]
        row = {
            "change_unit_id": f"cu_{uid:05d}", KEY_COL: rn,
            "change_type": "removed", "changed_field": "",
            "old_value": "", "new_value": "",
            "importance_score": 1.0,
        }
        row.update({c: soft_str(rec.get(c, "")) for c in COMPARE_COLS})
        rows.append(row); uid += 1

    # Modified
    for rn in sorted(set(prev_map) & set(cur_map)):
        prev_rec, cur_rec = prev_map[rn], cur_map[rn]
        for field in COMPARE_COLS:
            old_v = soft_str(prev_rec.get(field, ""))
            new_v = soft_str(cur_rec.get(field, ""))
            if old_v == new_v:
                continue
            base_row = {
                "change_unit_id": f"cu_{uid:05d}", KEY_COL: rn,
                "change_type": "modified", "changed_field": field,
                "old_value": old_v, "new_value": new_v,
            }
            base_row.update({c: soft_str(cur_rec.get(c, "")) for c in COMPARE_COLS})
            base_row["importance_score"] = compute_importance(base_row)
            rows.append(base_row); uid += 1

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df["importance_score"] = pd.to_numeric(df["importance_score"], errors="coerce").fillna(0.0)
    return df.sort_values("importance_score", ascending=False).reset_index(drop=True)

def summarize_change_stats(cu: pd.DataFrame) -> Dict[str, Any]:
    if len(cu) == 0:
        return {"total": 0, "added": 0, "removed": 0, "modified": 0, "recalls_affected": 0}
    return {
        "total": len(cu),
        "added":    int((cu["change_type"] == "added").sum()),
        "removed":  int((cu["change_type"] == "removed").sum()),
        "modified": int((cu["change_type"] == "modified").sum()),
        "recalls_affected": int(cu[KEY_COL].nunique()),
    }


## 4) ROUTE CLASSIFICATION

Route classification: Divide the change unit into routes such as new_igh_risk, consumption-action update, incident escalation, etc., and generate route_priority.

In [ ]:
# 4) ROUTE CLASSIFICATION

def _severity_score(row: Dict[str, Any]) -> float:
    text = " ".join([
        soft_str(row.get("Recall Heading", "")),
        soft_str(row.get("Hazard Description", "")),
        soft_str(row.get("Consumer Action", "")),
        soft_str(row.get("Incidents", "")),
    ]).lower()
    return sum(w for t, w in SEVERE_TERM_WEIGHTS.items() if t in text)

def _numeric_delta(field: str, old_v: str, new_v: str) -> float:
    if not any(k in field.lower() for k in ["incident", "injur", "unit"]):
        return 0.0
    old_n, new_n = safe_int_from_text(old_v), safe_int_from_text(new_v)
    return float(new_n - old_n) if old_n is not None and new_n is not None else 0.0

def assign_route(row: Dict[str, Any]) -> Dict[str, Any]:
    change_type = normalize_text(row.get("change_type", "")).lower()
    field       = normalize_text(row.get("changed_field", "")).lower()
    importance  = _num(row.get("importance_score", 0.0))
    severity    = _severity_score(row)
    delta       = _numeric_delta(field, row.get("old_value", ""), row.get("new_value", ""))
    action_text = " ".join([
        soft_str(row.get("Consumer Action", "")),
        soft_str(row.get("Remedy", "")),
    ]).lower()

    if change_type == "added" and (severity >= 5 or importance >= 5):
        route, base = "new_high_risk", 100
    elif "consumer action" in field or "remedy" in field or \
         re.search(r"stop use|refund|repair|replace|return", action_text):
        route, base = "consumer_action_update", 85
    elif any(k in field for k in ["incident", "injur", "unit", "hazard"]) or delta > 0:
        route, base = "incident_escalation", 75
    elif change_type == "removed":
        route, base = "removed_or_resolved", 45
    else:
        route, base = "metadata_or_context_update", 30

    priority = base + 1.5 * importance + 1.2 * severity + max(0.0, delta)
    actionability = (
        (1.0 if route in {"new_high_risk", "consumer_action_update"} else 0.0)
        + (1.0 if re.search(r"stop use|refund|repair|replace|immediately|return", action_text) else 0.0)
        + (0.5 if severity >= 5 else 0.0)
    )
    return {
        "route_label":        route,
        "route_priority":     round(priority, 3),
        "severity_signal":    round(severity, 3),
        "numeric_delta":      round(delta, 3),
        "actionability":      round(actionability, 3),
        "route_rank":         ROUTE_ORDER.get(route, 9),
    }

def augment_change_units(cu: pd.DataFrame) -> pd.DataFrame:
    if cu is None or len(cu) == 0:
        return pd.DataFrame()
    df = cu.copy()
    for c in ["changed_field", "importance_score"]:
        if c not in df.columns:
            df[c] = "" if c == "changed_field" else 0.0

    drop = [c for c in ["route_label","route_priority","severity_signal",
                         "numeric_delta","actionability","route_rank",
                         "canonical_recall_number","importance_weight"] if c in df.columns]
    if drop:
        df = df.drop(columns=drop)

    routes = df.apply(lambda r: pd.Series(assign_route(r.to_dict())), axis=1)
    df = pd.concat([df.reset_index(drop=True), routes.reset_index(drop=True)], axis=1)

    imp = pd.to_numeric(df["importance_score"], errors="coerce").fillna(0.0)
    sev = pd.to_numeric(df["severity_signal"],  errors="coerce").fillna(0.0)
    df["importance_weight"]      = (1 + np.ceil(np.clip(0.6 * imp + 0.25 * sev, 0, 5))).astype(int)
    df["canonical_recall_number"] = df[KEY_COL].map(canon_recall_number)

    return df.sort_values(
        ["route_rank", "route_priority", "importance_score"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

def build_route_digest(cu: pd.DataFrame, top_k: int = 3) -> List[Dict[str, Any]]:
    cu = augment_change_units(cu)
    if len(cu) == 0:
        return []
    rows = []
    for route, grp in cu.groupby("route_label", dropna=False):
        grp = grp.sort_values(["route_priority","importance_score"], ascending=[False,False]).head(top_k)
        recalls = [canon_recall_number(x) for x in grp[KEY_COL].astype(str) if canon_recall_number(x)]
        fields  = [normalize_text(x) for x in grp.get("changed_field", pd.Series(dtype=str)).astype(str) if normalize_text(x)]
        rows.append({
            "route_label":        normalize_text(route),
            "count":              int((cu["route_label"] == route).sum()),
            "top_recalls":        list(dict.fromkeys(recalls))[:top_k],
            "top_fields":         list(dict.fromkeys(fields))[:top_k],
            "avg_priority":       round(float(pd.to_numeric(grp["route_priority"], errors="coerce").mean()), 3),
        })
    return sorted(rows, key=lambda x: (-x["avg_priority"], x["route_label"]))


## 5) RETRIEVAL

Retrieval construction: Aggregate recall changes into candidate documents and use TF-IDF+hybrid score to retrieve the most relevant and important recall evidence.

In [ ]:
# 5) RETRIEVAL


def build_recall_candidate_table(cu: pd.DataFrame) -> pd.DataFrame:
    cache_key = (id(cu), len(cu))
    if cache_key in RECALL_CANDIDATE_CACHE:
        return RECALL_CANDIDATE_CACHE[cache_key].copy()

    if len(cu) == 0:
        return pd.DataFrame()

    work = augment_change_units(cu)
    rows = []

    for canon, grp in work.groupby("canonical_recall_number", dropna=False):
        canon = normalize_text(canon)
        if not canon:
            continue

        grp = grp.copy()
        grp["field_bonus"] = grp["changed_field"].map(KEY_FIELD_BONUS).fillna(0.0)
        grp["type_bonus"]  = grp["change_type"].map(
            {"modified": 3.0, "added": 2.0, "removed": 1.0}
        ).fillna(0.0)
        grp["rep_score"]   = grp["importance_score"] + grp["field_bonus"] + grp["type_bonus"]

        rep = grp.sort_values("rep_score", ascending=False).iloc[0]

        added    = bool((grp["change_type"] == "added").any())
        removed  = bool((grp["change_type"] == "removed").any())
        modified = bool((grp["change_type"] == "modified").any())

        original_numbers = sorted({normalize_text(x) for x in grp[KEY_COL].astype(str) if normalize_text(x)})
        changed_fields   = sorted({normalize_text(x) for x in grp["changed_field"].astype(str) if normalize_text(x)})
        change_types     = sorted({normalize_text(x) for x in grp["change_type"].astype(str) if normalize_text(x)})

        # Suppress obvious format-noise candidates
        format_carryover = added and removed and not modified and len(original_numbers) >= 2

        rep_text = " ".join([
            soft_str(rep.get("Recall Heading", "")), soft_str(rep.get("Name of product", "")),
            soft_str(rep.get("Hazard Description", "")), soft_str(rep.get("Consumer Action", "")),
            soft_str(rep.get("Remedy", "")), soft_str(rep.get("Incidents", "")),
        ]).lower()

        severe_bonus = sum(w for t, w in SEVERE_TERM_WEIGHTS.items() if t in rep_text)
        field_bonus  = sum(KEY_FIELD_BONUS.get(f, 0.0) for f in changed_fields)
        summary_score = float(grp["importance_score"].max()) + severe_bonus + field_bonus
        if added:    summary_score += 2.0
        if modified: summary_score += 1.0
        if format_carryover: summary_score -= 20.0

        # Collect old/new value pairs per changed field for contrastive generation
        old_new_pairs = []
        for _, r in grp[grp["change_type"] == "modified"].iterrows():
            f  = normalize_text(r.get("changed_field", ""))
            ov = normalize_text(r.get("old_value", ""))
            nv = normalize_text(r.get("new_value", ""))
            if f and (ov or nv):
                old_new_pairs.append({"field": f, "old": ov[:300], "new": nv[:300]})

        doc_parts = [
            f"Recall Number: {canon}",
            f"Change Types: {' | '.join(change_types)}",
            f"Changed Fields: {' | '.join(changed_fields)}",
            f"Recall Heading: {soft_str(rep.get('Recall Heading', ''))}",
            f"Name of product: {soft_str(rep.get('Name of product', ''))}",
            f"Hazard Description: {soft_str(rep.get('Hazard Description', ''))}",
            f"Consumer Action: {soft_str(rep.get('Consumer Action', ''))}",
            f"Remedy: {soft_str(rep.get('Remedy', ''))}",
            f"Incidents: {soft_str(rep.get('Incidents', ''))}",
            f"Manufactured In: {soft_str(rep.get('Manufactured In', ''))}",
            f"Units: {soft_str(rep.get('Units', ''))}",
        ]
        if old_new_pairs:
            doc_parts.append(f"Old→New Changes: {json.dumps(old_new_pairs, ensure_ascii=False)}")

        rows.append({
            "doc_id":                 f"recall::{canon}",
            "recall_number":          canon,
            "display_recall_number":  canon,
            "doc_type":               "recall_candidate",
            "text":                   "\n".join(doc_parts),
            "importance_score":       round(summary_score, 3),
            "summary_score":          round(summary_score, 3),
            "max_importance":         float(grp["importance_score"].max()),
            "format_carryover":       bool(format_carryover),
            "change_types":           " | ".join(change_types),
            "changed_fields":         " | ".join(changed_fields),
            "original_recall_numbers": " | ".join(original_numbers),
            "old_new_pairs":          json.dumps(old_new_pairs, ensure_ascii=False),
            # Route info from highest-priority change unit for this recall
            "route_label":            normalize_text(rep.get("route_label", "")),
            "route_priority":         _num(rep.get("route_priority", 0.0)),
        })

    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    out = out.sort_values(["summary_score", "recall_number"], ascending=[False, True]).reset_index(drop=True)
    RECALL_CANDIDATE_CACHE[cache_key] = out.copy()
    return out

def build_retrieval_corpus(cu: pd.DataFrame) -> pd.DataFrame:
    candidates = build_recall_candidate_table(cu)
    if len(candidates) == 0:
        return pd.DataFrame()
    return candidates[~candidates["format_carryover"]].reset_index(drop=True)


class SimpleRetriever:
    def __init__(self, corpus_df: pd.DataFrame):
        self.corpus_df = corpus_df.copy()
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.X = None
        self._fit()

    def _fit(self):
        if len(self.corpus_df) == 0:
            return
        texts = self.corpus_df["text"].fillna("").astype(str).tolist()
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")
        self.X = self.vectorizer.fit_transform(texts)

    def _norm(self, s: pd.Series) -> pd.Series:
        s = pd.to_numeric(s, errors="coerce").fillna(0.0)
        mn, mx = float(s.min()), float(s.max())
        return pd.Series(np.zeros(len(s)), index=s.index) if abs(mx - mn) < 1e-9 \
               else (s - mn) / (mx - mn + 1e-9)

    def search(self, query: str, top_k: int = 8, intent: str = "generic") -> pd.DataFrame:
        if self.vectorizer is None or len(self.corpus_df) == 0:
            return self.corpus_df.head(0).copy()

        q    = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.X).reshape(-1)
        out  = self.corpus_df.copy()
        out["retrieval_score"]  = sims
        out["importance_norm"]  = self._norm(out.get("summary_score", out.get("importance_score", 0.0)))

        q_toks = {t.lower() for t in re.findall(r"[A-Za-z][A-Za-z\-]{3,}", query)}
        def _overlap(txt: str) -> float:
            toks = {t.lower() for t in re.findall(r"[A-Za-z][A-Za-z\-]{3,}", normalize_text(txt))}
            return len(q_toks & toks) / max(len(q_toks), 1) if q_toks else 0.0
        out["token_overlap"] = out["text"].map(_overlap)

        cf  = out.get("changed_fields", "").fillna("").astype(str)
        ct  = out.get("change_types",   "").fillna("").astype(str)
        txt = out["text"].fillna("").astype(str).str.lower()
        bonus = np.zeros(len(out), dtype=float)

        if intent == "added_high":
            bonus += ct.str.contains("added", regex=False).astype(float) * 0.18
        elif intent == "consumer_action":
            bonus += ct.str.contains("modified", regex=False).astype(float) * 0.15
            bonus += cf.str.contains("Consumer Action", regex=False).astype(float) * 0.45
            bonus += cf.str.contains("Remedy", regex=False).astype(float) * 0.18
        elif intent == "incidents_units":
            bonus += cf.str.contains("Incidents|Units", regex=True, na=False).astype(float) * 0.18
            bonus += cf.str.contains("Hazard", regex=False, na=False).astype(float) * 0.08
        elif intent == "removed_legacy":
            bonus += ct.str.contains("removed", regex=False).astype(float) * 0.20

        # Weight: retrieval 0.30, importance 0.40, overlap 0.10, intent bonus
        out["hybrid_score"] = (
            0.30 * out["retrieval_score"]
            + 0.40 * out["importance_norm"]
            + 0.10 * out["token_overlap"]
            + bonus
        )
        return out.sort_values(["hybrid_score","summary_score"], ascending=[False,False]) \
                  .head(top_k).reset_index(drop=True)


def pick_top_hazard_terms(cu: pd.DataFrame, top_n: int = 8) -> List[str]:
    if len(cu) == 0:
        return []
    top = cu.sort_values("importance_score", ascending=False).head(300)
    text = " ".join(top.get("Hazard Description", pd.Series(dtype=str)).fillna("").astype(str)).lower()
    return [t for t in SEVERE_TERM_WEIGHTS if t in text][:top_n]

def build_query_bundle(cu: pd.DataFrame) -> List[Dict[str, str]]:
    """Build data-driven queries from route classification."""
    cu_aug = augment_change_units(cu)
    hazard_str = " ".join(pick_top_hazard_terms(cu_aug, top_n=8)) or \
                 "fire battery burn choking injury refund stop use"

    route_templates = {
        "new_high_risk":            "newly added recalls severe hazards {hazards} child infant fire burn choking battery high priority",
        "consumer_action_update":   "consumer action remedy changed stop use refund repair replace return immediately",
        "incident_escalation":      "incidents injuries units affected hazard worsened increased reports expanded scope",
        "removed_or_resolved":      "removed recalls resolved legacy entries previous snapshot differences",
        "metadata_or_context_update": "descriptive updates manufacturing heading description context changed",
    }
    intent_map = {
        "new_high_risk":            "added_high",
        "consumer_action_update":   "consumer_action",
        "incident_escalation":      "incidents_units",
        "removed_or_resolved":      "removed_legacy",
        "metadata_or_context_update": "generic",
    }

    queries, seen = [], set()
    for route, grp in cu_aug.groupby("route_label", dropna=False):
        grp = grp.sort_values(["route_priority","importance_score"], ascending=[False,False]).head(4)
        recall_terms = " ".join(
            [canon_recall_number(x) for x in grp[KEY_COL].astype(str).tolist()[:3] if canon_recall_number(x)]
        )
        q = route_templates.get(route, "{hazards} important recall changes").format(hazards=hazard_str)
        if recall_terms:
            q += f" {recall_terms}"
        key = (intent_map.get(route, "generic"), q)
        if key in seen or not q:
            continue
        seen.add(key)
        queries.append({"name": normalize_text(route), "intent": intent_map.get(route, "generic"), "query": q})

    # Catch-all high-priority mix
    top_recalls = " ".join([
        canon_recall_number(x) for x in
        cu_aug.sort_values("route_priority", ascending=False)[KEY_COL].head(5).astype(str)
        if canon_recall_number(x)
    ])
    queries.append({"name": "priority_mix", "intent": "generic",
                    "query": f"highest priority recall changes {hazard_str} {top_recalls}".strip()})
    return queries

def hybrid_search_queries(retriever: SimpleRetriever,
                          query_specs: List[Dict[str, str]],
                          per_query_k: int = 8,
                          final_k: int = 10) -> pd.DataFrame:
    if not query_specs:
        return retriever.corpus_df.head(0).copy()

    quota = max(1, final_k // max(len(query_specs), 1))
    selected, seen = [], set()

    for spec in query_specs:
        tmp = retriever.search(spec.get("query",""), top_k=per_query_k, intent=spec.get("intent","generic"))
        for _, row in tmp.head(quota).iterrows():
            rid = row["doc_id"]
            if rid not in seen:
                seen.add(rid)
                r = row.to_dict()
                r["query_name"] = spec.get("name","")
                selected.append(r)
            if len(selected) >= final_k:
                break
        if len(selected) >= final_k:
            break

    # Fill remaining slots
    if len(selected) < final_k:
        pool = {}
        for spec in query_specs:
            tmp = retriever.search(spec.get("query",""), top_k=per_query_k, intent=spec.get("intent","generic"))
            for _, row in tmp.iterrows():
                rid = row["doc_id"]
                rd  = row.to_dict()
                if pool.get(rid) is None or _num(rd.get("hybrid_score")) > _num(pool[rid].get("hybrid_score")):
                    pool[rid] = rd
        extras = pd.DataFrame(list(pool.values())) if pool else pd.DataFrame()
        if len(extras):
            extras = extras[~extras["doc_id"].isin(seen)].sort_values(
                ["hybrid_score","summary_score"], ascending=[False,False]
            )
            for _, row in extras.iterrows():
                selected.append(row.to_dict())
                if len(selected) >= final_k:
                    break

    if not selected:
        return retriever.corpus_df.head(0).copy()
    return pd.DataFrame(selected) \
             .sort_values(["hybrid_score","summary_score"], ascending=[False,False]) \
             .head(final_k).reset_index(drop=True)


## 6) SHORTLIST HELPERS

Shortlist and evidence auxiliary functions: Construct a summary shortlist from the retrieval result to ensure that the summary covers key recalls.

In [ ]:
# 6) SHORTLIST HELPERS


def extract_field_from_doc(doc_text: str, label: str) -> str:
    m = re.search(rf"(?:^|\n){re.escape(label)}:\s*(.*?)(?:\n|$)",
                  normalize_text(doc_text), flags=re.I)
    return normalize_text(m.group(1)) if m else ""

def build_shortlist_from_retrieved(retrieved: pd.DataFrame, max_items: int = 8) -> List[Dict[str, Any]]:
    if retrieved is None or len(retrieved) == 0:
        return []
    work = retrieved.copy()
    for c in ["hybrid_score","summary_score","retrieval_score","importance_score"]:
        if c not in work.columns:
            work[c] = 0.0
    work = work.sort_values(
        ["hybrid_score","summary_score","retrieval_score","importance_score"],
        ascending=[False]*4
    )
    rows, seen = [], set()
    for _, r in work.iterrows():
        recall = canon_recall_number(r.get("recall_number",""))
        if not recall or recall in seen:
            continue
        seen.add(recall)
        doc   = normalize_text(r.get("text",""))
        pairs = []
        try:
            pairs = json.loads(r.get("old_new_pairs", "[]") or "[]")
        except Exception:
            pass
        rows.append({
            "recall_number":    recall,
            "change_types":     normalize_text(r.get("change_types","")),
            "changed_fields":   normalize_text(r.get("changed_fields","")),
            "recall_heading":   extract_field_from_doc(doc, "Recall Heading"),
            "hazard_description": extract_field_from_doc(doc, "Hazard Description"),
            "consumer_action":  extract_field_from_doc(doc, "Consumer Action"),
            "remedy":           extract_field_from_doc(doc, "Remedy"),
            "incidents":        extract_field_from_doc(doc, "Incidents"),
            "units":            extract_field_from_doc(doc, "Units"),
            "manufactured_in":  extract_field_from_doc(doc, "Manufactured In"),
            "old_new_pairs":    pairs,
            "route_label":      normalize_text(r.get("route_label","")),
            "summary_score":    _num(r.get("summary_score", 0.0)),
            "hybrid_score":     _num(r.get("hybrid_score", 0.0)),
        })
        if len(rows) >= max_items:
            break
    return rows

def build_anchor_shortlist(corpus: pd.DataFrame, max_items: int = 6) -> List[Dict[str, Any]]:
    if corpus is None or len(corpus) == 0:
        return []
    work = corpus.copy()
    if "format_carryover" in work.columns:
        work = work[~work["format_carryover"].fillna(False)]
    if len(work) == 0:
        return []
    cf = work.get("changed_fields","").fillna("").astype(str).str.lower()
    ct = work.get("change_types","").fillna("").astype(str).str.lower()
    work["anchor_score"] = (
        pd.to_numeric(work.get("summary_score",0), errors="coerce").fillna(0.0)
        + cf.str.contains("consumer action|remedy|incidents|units|hazard", regex=True).astype(float) * 2.0
        + ct.str.contains("modified").astype(float) * 1.2
        + ct.str.contains("added").astype(float) * 1.0
    )
    work = work.sort_values("anchor_score", ascending=False)
    rows, seen = [], set()
    for _, r in work.iterrows():
        recall = canon_recall_number(r.get("recall_number",""))
        if not recall or recall in seen:
            continue
        seen.add(recall)
        doc   = normalize_text(r.get("text",""))
        pairs = []
        try:
            pairs = json.loads(r.get("old_new_pairs","[]") or "[]")
        except Exception:
            pass
        rows.append({
            "recall_number":    recall,
            "change_types":     normalize_text(r.get("change_types","")),
            "changed_fields":   normalize_text(r.get("changed_fields","")),
            "recall_heading":   extract_field_from_doc(doc, "Recall Heading"),
            "hazard_description": extract_field_from_doc(doc, "Hazard Description"),
            "consumer_action":  extract_field_from_doc(doc, "Consumer Action"),
            "remedy":           extract_field_from_doc(doc, "Remedy"),
            "incidents":        extract_field_from_doc(doc, "Incidents"),
            "units":            extract_field_from_doc(doc, "Units"),
            "old_new_pairs":    pairs,
            "route_label":      normalize_text(r.get("route_label","")),
            "summary_score":    _num(r.get("summary_score",0.0)),
            "hybrid_score":     _num(r.get("hybrid_score",0.0)),
        })
        if len(rows) >= max_items:
            break
    return rows

def merge_shortlists(primary: List, secondary: List, max_items: int = 10) -> List[Dict[str, Any]]:
    rows, seen = [], set()
    for src in [primary or [], secondary or []]:
        for row in src:
            recall = canon_recall_number(row.get("recall_number",""))
            if not recall or recall in seen:
                continue
            seen.add(recall)
            rows.append({**row, "recall_number": recall})
            if len(rows) >= max_items:
                return rows
    return rows

def ensure_evidence_for_shortlist(retrieved: pd.DataFrame, corpus: pd.DataFrame,
                                  shortlist: List, max_docs: int = 12) -> pd.DataFrame:
    base = retrieved.copy() if retrieved is not None and len(retrieved) > 0 else pd.DataFrame()
    if corpus is None or len(corpus) == 0:
        return base.head(max_docs)

    have = {canon_recall_number(x) for x in
            (base.get("recall_number", pd.Series(dtype=str)).astype(str).tolist() if len(base) else [])
            if canon_recall_number(x)}

    corp = corpus.copy()
    corp["_canon"] = corp.get("recall_number","").map(canon_recall_number)
    for c in ["hybrid_score","summary_score","importance_score"]:
        if c not in corp.columns:
            corp[c] = 0.0

    extras = []
    for item in shortlist:
        recall = canon_recall_number(item.get("recall_number",""))
        if not recall or recall in have:
            continue
        cand = corp[corp["_canon"] == recall].sort_values("summary_score", ascending=False)
        if len(cand) == 0:
            continue
        extras.append(cand.iloc[0].drop(labels=["_canon"]).to_dict())
        have.add(recall)
        if len(base) + len(extras) >= max_docs:
            break

    if extras:
        combo = pd.concat([base, pd.DataFrame(extras)], ignore_index=True, sort=False)
    else:
        combo = base.copy()

    for c in ["hybrid_score","summary_score","importance_score"]:
        if c not in combo.columns:
            combo[c] = 0.0

    return combo.sort_values(["hybrid_score","summary_score"], ascending=[False,False]) \
                .drop_duplicates(subset=["doc_id"], keep="first") \
                .head(max_docs).reset_index(drop=True)

def shortlist_to_json(shortlist: List, include_old_new: bool = False) -> str:
    compact = []
    for x in shortlist:
        item = {
            "recall_number":    x.get("recall_number",""),
            "change_types":     x.get("change_types",""),
            "changed_fields":   x.get("changed_fields",""),
            "recall_heading":   x.get("recall_heading",""),
            "hazard_description": x.get("hazard_description",""),
            "consumer_action":  x.get("consumer_action",""),
            "remedy":           x.get("remedy",""),
            "incidents":        x.get("incidents",""),
        }
        if include_old_new and x.get("old_new_pairs"):
            item["old_new_changes"] = x["old_new_pairs"]
        compact.append(item)
    return json.dumps(compact, ensure_ascii=False, indent=2)

def row_to_evidence_bullet(row: Dict[str, Any]) -> str:
    recall = canon_recall_number(row.get("recall_number","")) or ""
    if not recall:
        return ""
    parts = []
    if row.get("recall_heading"):   parts.append(row["recall_heading"])
    if row.get("change_types"):     parts.append(f"Change: {row['change_types']}.")
    if row.get("changed_fields"):   parts.append(f"Fields: {row['changed_fields']}.")
    if row.get("hazard_description"): parts.append(f"Hazard: {row['hazard_description']}.")
    if row.get("consumer_action"):  parts.append(f"Action: {row['consumer_action']}.")
    elif row.get("remedy"):         parts.append(f"Remedy: {row['remedy']}.")
    inc = row.get("incidents","")
    if inc and inc.lower() not in {"none reported","none",""}:
        parts.append(f"Incidents: {inc}.")
    return f"Recall {recall}: " + (" ".join(parts).strip() or "Evidence-backed recall update.")

def enforce_shortlist_coverage(text: str, retrieved: pd.DataFrame,
                               min_unique: int = 6, max_unique: int = 8,
                               preferred: Optional[List[str]] = None) -> str:
    shortlist = build_shortlist_from_retrieved(retrieved, max_items=max_unique)
    if not shortlist:
        return text
    cited    = {canon_recall_number(x) for x in extract_recall_numbers(text) if canon_recall_number(x)}
    pref_set = {canon_recall_number(x) for x in (preferred or []) if canon_recall_number(x)}
    ordered  = sorted(shortlist,
                      key=lambda r: (0 if canon_recall_number(r.get("recall_number","")) in pref_set else 1,
                                     -r.get("hybrid_score",0.0)))
    additions, count = [], len(cited)
    for row in ordered:
        if count >= max_unique:
            break
        recall = canon_recall_number(row.get("recall_number",""))
        if not recall or recall in cited:
            continue
        bullet = row_to_evidence_bullet(row)
        if bullet:
            additions.append("- " + bullet)
            cited.add(recall)
            count += 1
    return (normalize_text(text).strip() + "\n" + "\n".join(additions)).strip() if additions else text.strip()


## 7.0) Generation shared helper

The post-processing helper shared by the four generation methods: save the raw summary and generate the final summary through enforce_sthortlist_comoverage.

In [ ]:
# 7) FOUR GENERATION METHODS

def _apply_enforce(text: str, evidence_df: pd.DataFrame,
                   shortlist: List, max_unique: int = 10) -> Tuple[str, str]:
    raw = text
    final = enforce_shortlist_coverage(
        text, evidence_df,
        min_unique=min(7, len(shortlist)),
        max_unique=max_unique or 10,
        preferred=[x["recall_number"] for x in shortlist[:8]],
    )
    return raw, final


## 7.1) Method 1 Prompt-Only


In [ ]:
# Method 1: Prompt-Only

def generate_prompt_only(cu: pd.DataFrame,
                         corpus: Optional[pd.DataFrame] = None
                         ) -> Tuple[str, str, Dict[str, Any]]:
    """Returns (raw_summary, final_summary, meta)."""
    corpus = corpus if corpus is not None else build_retrieval_corpus(cu)
    brief  = build_shortlist_from_retrieved(
        corpus.assign(hybrid_score=corpus.get("summary_score",0)).head(10), max_items=10
    ) if len(corpus) > 0 else []

    if LLM_MODE == "offline" or not brief:
        lines = ["Priority recall watchlist:"]
        for r in brief:
            ev = normalize_text(r.get("recall_heading","") + " " + r.get("hazard_description",""))[:300]
            lines.append(f"- Recall {r['recall_number']}: type={r['change_types']}; fields={r['changed_fields']}. {ev}")
        raw = "\n".join(lines) if lines else "No meaningful changes detected."
        return raw, raw, {"mode": "offline", "usage": {}}

    prompt = (
        "You are analyzing CPSC recall updates. Based only on the structured candidates below, "
        "write 6–10 bullet points. Each bullet MUST start with 'Recall <Number>:' then explain "
        "what changed, why it matters, and what action is needed. Be evidence-grounded.\n\n"
        + json.dumps([{
            "recall_number": r["recall_number"],
            "change_types": r["change_types"],
            "changed_fields": r["changed_fields"],
            "hazard_description": r["hazard_description"],
            "consumer_action": r["consumer_action"],
          } for r in brief], ensure_ascii=False, indent=2)
    )
    text, usage = call_llm([
        {"role": "system", "content": "Be factual, concise, always cite recall numbers."},
        {"role": "user",   "content": prompt},
    ])
    raw, final = _apply_enforce(text, corpus, brief, max_unique=len(brief) or 10)
    return raw, final, {"mode": "llm", "usage": usage}


## 7.2) Method 2 RAG

In [ ]:
# Method 2: RAG

def generate_rag(cu: pd.DataFrame,
                 retriever: SimpleRetriever
                 ) -> Tuple[str, str, pd.DataFrame, Dict[str, Any]]:
    """Returns (raw_summary, final_summary, evidence_df, meta)."""
    query_specs = build_query_bundle(cu)
    retrieved   = hybrid_search_queries(retriever, query_specs, per_query_k=8, final_k=RAG_MAX_DOCS)

    ret_sl   = build_shortlist_from_retrieved(retrieved, max_items=8)
    anc_sl   = build_anchor_shortlist(retriever.corpus_df, max_items=6)
    shortlist = merge_shortlists(ret_sl, anc_sl, max_items=RAG_MAX_DOCS)
    evidence  = ensure_evidence_for_shortlist(retrieved, retriever.corpus_df, shortlist, max_docs=RAG_MAX_DOCS)

    if LLM_MODE == "offline":
        lines = ["Evidence-backed recall watchlist:"]
        for row in shortlist:
            lines.append("- " + row_to_evidence_bullet(row))
        raw, final = _apply_enforce("\n".join(lines), evidence, shortlist, max_unique=len(shortlist) or 10)
        return raw, final, evidence, {"mode": "offline", "usage": {}, "query_specs": query_specs}

    ev_texts = evidence["text"].fillna("").astype(str).tolist()
    prompt = (
        "Use only the evidence below to write 7–10 bullet points summarizing the most important recall changes. "
        "Every bullet MUST start with 'Recall <Number>:' and cover exactly one recall. "
        "Cover as many SHORTLIST recalls as evidence supports. "
        "Prioritize: new high-risk recalls → consumer-action/remedy changes → incident/unit updates. "
        "Mention changed fields. Do not invent facts.\n\n"
        "SHORTLIST:\n" + shortlist_to_json(shortlist)
        + "\n\nEVIDENCE:\n" + "\n\n---\n\n".join(ev_texts)
    )
    text, usage = call_llm([
        {"role": "system", "content": "Factual, concise, one bullet per recall, cite recall numbers."},
        {"role": "user",   "content": prompt},
    ])
    raw, final = _apply_enforce(text, evidence, shortlist, max_unique=len(shortlist) or 10)
    return raw, final, evidence, {"mode": "llm", "usage": usage, "query_specs": query_specs, "shortlist": shortlist}


## 7.3) Method 3 Agentic RAG


In [ ]:
# Method 3: Agentic RAG

def generate_agentic_rag(cu: pd.DataFrame,
                         retriever: SimpleRetriever
                         ) -> Tuple[str, str, pd.DataFrame, Dict[str, Any]]:
    """Returns (raw_summary, final_summary, evidence_df, meta)."""
    seed_queries = build_query_bundle(cu)

    if LLM_MODE == "offline":
        retrieved = hybrid_search_queries(retriever, seed_queries, per_query_k=8, final_k=AGENT_MAX_DOCS)
        ret_sl    = build_shortlist_from_retrieved(retrieved, max_items=8)
        anc_sl    = build_anchor_shortlist(retriever.corpus_df, max_items=6)
        shortlist = merge_shortlists(ret_sl, anc_sl, max_items=AGENT_MAX_DOCS)
        evidence  = ensure_evidence_for_shortlist(retrieved, retriever.corpus_df, shortlist, max_docs=AGENT_MAX_DOCS)
        lines     = ["Agentic recall watchlist:"]
        for row in shortlist:
            lines.append("- " + row_to_evidence_bullet(row))
        raw, final = _apply_enforce("\n".join(lines), evidence, shortlist, max_unique=len(shortlist) or 10)
        return raw, final, evidence, {"mode": "offline", "usage": {}, "queries": [x["query"] for x in seed_queries]}

    # Step 1: LLM planning — generate additional queries
    plan_text, u1 = call_llm([
        {"role": "system", "content": "Return only valid JSON."},
        {"role": "user",   "content": (
            "Build a retrieval plan for CPSC recall-change analysis. "
            "Return JSON: {\"queries\": [\"query1\", \"query2\", ...]} "
            "covering: (1) new high-severity recalls, (2) consumer action/remedy changes, "
            "(3) incident/unit escalations."
        )},
    ])
    llm_queries = []
    try:
        llm_queries = json.loads(plan_text).get("queries", [])
        if not isinstance(llm_queries, list):
            llm_queries = []
    except Exception:
        pass

    intent_rules = [
        (r"consumer|remedy|refund", "consumer_action"),
        (r"incident|unit",          "incidents_units"),
        (r"added|new",              "added_high"),
        (r"removed|legacy",         "removed_legacy"),
    ]
    query_specs = list(seed_queries)
    for i, q in enumerate(llm_queries[:4], 1):
        qq = normalize_text(q)
        if not qq:
            continue
        intent = next((v for pat, v in intent_rules if re.search(pat, qq.lower())), "generic")
        query_specs.append({"name": f"llm_{i}", "intent": intent, "query": qq})

    retrieved = hybrid_search_queries(retriever, query_specs, per_query_k=8, final_k=AGENT_MAX_DOCS)
    ret_sl    = build_shortlist_from_retrieved(retrieved, max_items=8)
    anc_sl    = build_anchor_shortlist(retriever.corpus_df, max_items=6)
    shortlist = merge_shortlists(ret_sl, anc_sl, max_items=AGENT_MAX_DOCS)
    evidence  = ensure_evidence_for_shortlist(retrieved, retriever.corpus_df, shortlist, max_docs=AGENT_MAX_DOCS)
    ev_texts  = evidence["text"].fillna("").astype(str).tolist()

    # Step 2: Selection
    sel_text, u2 = call_llm([
        {"role": "system", "content": "Return only valid JSON."},
        {"role": "user",   "content": (
            "From the shortlist below, select 7–8 recalls to cover in a grounded summary. "
            "Prefer recalls with clear hazard, consumer-action, remedy, incident, or newly-added signals. "
            "Return JSON: {\"selected_recalls\": [\"XX-XXX\", ...]}\n\n"
            + shortlist_to_json(shortlist)
        )},
    ])
    selected_recalls = [x["recall_number"] for x in shortlist[:8]]
    try:
        picked = [canon_recall_number(x) for x in json.loads(sel_text).get("selected_recalls",[]) if canon_recall_number(x)]
        if picked:
            selected_recalls = list(dict.fromkeys(picked + selected_recalls))[:8]
    except Exception:
        pass

    # Step 3: Draft
    draft, u3 = call_llm([
        {"role": "system", "content": "Factual, concise, one bullet per recall, cite recall numbers."},
        {"role": "user",   "content": (
            "Using only the evidence below, write 7–10 bullet points. "
            "Every bullet starts with 'Recall <Number>:'. Cover SELECTED_RECALLS first.\n\n"
            f"SELECTED_RECALLS: {json.dumps(selected_recalls)}\n\n"
            "SHORTLIST:\n" + shortlist_to_json(shortlist)
            + "\n\nEVIDENCE:\n" + "\n\n---\n\n".join(ev_texts)
        )},
    ])

    # Step 4: Revise
    revised, u4 = call_llm([
        {"role": "system", "content": "Return a grounded final answer only."},
        {"role": "user",   "content": (
            "Revise the draft: remove unsupported claims, improve SELECTED_RECALLS coverage. "
            "Keep format: each bullet starts with 'Recall <Number>:'.\n\n"
            f"SELECTED_RECALLS: {json.dumps(selected_recalls)}\n\n"
            "EVIDENCE:\n" + "\n\n---\n\n".join(ev_texts[:10])
            + "\n\nDRAFT:\n" + draft
        )},
    ])

    usage = {}
    for d in [u1, u2, u3, u4]:
        for k, v in d.items():
            if isinstance(v, (int, float)):
                usage[k] = usage.get(k, 0) + v

    raw, final = _apply_enforce(revised, evidence, shortlist, max_unique=len(shortlist) or 10)
    return raw, final, evidence, {
        "mode": "llm", "usage": usage,
        "plan": {"queries": llm_queries},
        "queries": [x["query"] for x in query_specs],
        "shortlist": shortlist,
        "selected_recalls": selected_recalls,
    }


## 7.4) Method 4 Contrastive Adaptive RAG

In [ ]:
# Method 4: Contrastive Adaptive RAG

def _build_old_new_pairs_for_shortlist(shortlist: List[Dict]) -> List[Dict[str, Any]]:
    """Extract structured old→new value pairs from shortlist for contrastive prompting."""
    result = []
    for item in shortlist:
        pairs = item.get("old_new_pairs", [])
        if not pairs and item.get("change_types","") == "added":
            pairs = []  # New recall — no old values

        entry = {
            "recall_number":    item.get("recall_number",""),
            "route":            item.get("route_label",""),
            "change_types":     item.get("change_types",""),
            "recall_heading":   item.get("recall_heading",""),
            "hazard_description": item.get("hazard_description",""),
            "consumer_action":  item.get("consumer_action",""),
            "remedy":           item.get("remedy",""),
            "incidents":        item.get("incidents",""),
            "old_new_changes":  pairs,  # list of {field, old, new}
        }
        result.append(entry)
    return result

def _contrastive_generation_prompt(structured_pairs: List[Dict],
                                   evidence_texts: List[str],
                                   route_digest: List[Dict]) -> str:
    return (
        "You are writing a CHANGE-TYPE-AWARE, EVIDENCE-GROUNDED summary for CPSC recall monitoring.\n\n"
        "## Your task\n"
        "Write 7–10 bullet points following this EXACT format for each recall:\n\n"
        "For MODIFIED recalls (has old→new changes):\n"
        "  Recall XX-XXX [ROUTE]: [Product] — previously [old value summary], "
        "now updated to [new value summary]. Consumer impact: [why this matters].\n\n"
        "For ADDED recalls (new recall, no previous):\n"
        "  Recall XX-XXX [NEW HIGH RISK]: [Product] — newly added recall. "
        "Hazard: [hazard]. Action required: [consumer action].\n\n"
        "For REMOVED recalls:\n"
        "  Recall XX-XXX [REMOVED]: [Product] — removed from current snapshot. "
        "Previously: [brief description].\n\n"
        "## Rules\n"
        "1. Only write a bullet if the old/new values OR hazard/action details appear in EVIDENCE.\n"
        "2. If you cannot find both old and new values in evidence for a modified recall, "
        "write what is supported and append [partial evidence].\n"
        "3. Do NOT invent facts. Do NOT combine multiple recalls in one bullet.\n"
        "4. Cover highest-priority routes first: new_high_risk → consumer_action_update → "
        "incident_escalation → removed_or_resolved.\n\n"
        "## Route digest (for priority ordering)\n"
        + json.dumps(route_digest, ensure_ascii=False, indent=2)
        + "\n\n## Structured recall changes (with old→new values)\n"
        + json.dumps(structured_pairs, ensure_ascii=False, indent=2)
        + "\n\n## Evidence\n"
        + "\n\n---\n\n".join(evidence_texts)
    )

def _self_verify_prompt(draft: str, evidence_texts: List[str]) -> str:
    return (
        "You are verifying a recall-change summary for factual grounding.\n\n"
        "For each bullet in the DRAFT, classify every factual claim:\n"
        "- 'fully_supported': claim text closely matches evidence\n"
        "- 'partially_supported': claim is plausible but only partially matches evidence\n"
        "- 'unsupported': claim has no matching text in evidence\n\n"
        "Return ONLY valid JSON:\n"
        "{\n"
        "  \"bullets\": [\n"
        "    {\n"
        "      \"recall_number\": \"XX-XXX\",\n"
        "      \"route\": \"...\",\n"
        "      \"bullet_text\": \"...\",\n"
        "      \"claims\": [\n"
        "        {\"text\": \"...\", \"verdict\": \"fully_supported\"|\"partially_supported\"|\"unsupported\","
        " \"evidence_snippet\": \"...\"}\n"
        "      ],\n"
        "      \"overall_verdict\": \"fully_supported\"|\"partially_supported\"|\"unsupported\",\n"
        "      \"keep\": true|false\n"
        "    }\n"
        "  ]\n"
        "}\n\n"
        "Set keep=false only if ALL claims are unsupported.\n\n"
        "EVIDENCE:\n" + "\n\n---\n\n".join(evidence_texts[:10])
        + "\n\nDRAFT:\n" + draft
    )

def generate_contrastive_adaptive_rag(cu: pd.DataFrame,
                                      retriever: SimpleRetriever
                                      ) -> Tuple[str, str, pd.DataFrame, Dict[str, Any]]:
    """
    Method 4: Contrastive Adaptive RAG.
    Key innovations:
    - Uses old_value/new_value pairs to force structured contrastive bullet format.
    - Route classification drives both retrieval AND generation structure.
    - Self-verification loop produces three-tier claim grounding (fully/partially/unsupported)
      as a by-product of generation, not a separate eval step.
    Returns (raw_summary, final_summary, evidence_df, meta).
    """
    cu_aug      = augment_change_units(cu)
    query_specs = build_query_bundle(cu_aug)
    retrieved   = hybrid_search_queries(retriever, query_specs, per_query_k=8,
                                        final_k=max(RAG_MAX_DOCS, 12))

    ret_sl    = build_shortlist_from_retrieved(retrieved, max_items=8)
    anc_sl    = build_anchor_shortlist(retriever.corpus_df, max_items=6)
    shortlist = merge_shortlists(ret_sl, anc_sl, max_items=max(RAG_MAX_DOCS, 12))
    evidence  = ensure_evidence_for_shortlist(retrieved, retriever.corpus_df, shortlist,
                                              max_docs=max(RAG_MAX_DOCS, 12))
    ev_texts  = evidence["text"].fillna("").astype(str).tolist()
    route_digest = build_route_digest(cu_aug)

    if LLM_MODE == "offline":
        lines = ["Contrastive recall watchlist:"]
        for item in shortlist:
            recall = item.get("recall_number","")
            pairs  = item.get("old_new_pairs",[])
            bullet = f"Recall {recall}"
            if pairs:
                p = pairs[0]
                bullet += f": [{p.get('field','')}] was '{p.get('old','')}', now '{p.get('new','')}'"
            else:
                bullet += ": " + row_to_evidence_bullet(item)
            lines.append("- " + bullet)
        raw, final = _apply_enforce("\n".join(lines), evidence, shortlist,
                                    max_unique=len(shortlist) or 10)
        return raw, final, evidence, {"mode": "offline", "usage": {}, "route_digest": route_digest}

    structured_pairs = _build_old_new_pairs_for_shortlist(shortlist)

    # Step 1: Contrastive generation
    draft, u1 = call_llm([
        {"role": "system", "content": (
            "Be factual and change-aware. Always use the contrastive format: "
            "'previously X, now Y'. Cite recall numbers. Prefer evidence-grounded old→new comparisons."
        )},
        {"role": "user", "content": _contrastive_generation_prompt(
            structured_pairs, ev_texts, route_digest
        )},
    ])

    # Step 2: Self-verification — produces three-tier grounding as a by-product
    verify_text, u2 = call_llm([
        {"role": "system", "content": "Return only valid JSON. No markdown."},
        {"role": "user",   "content": _self_verify_prompt(draft, ev_texts)},
    ])

    verification_result = {}
    final_bullets = []
    try:
        verification_result = json.loads(verify_text)
        bullets = verification_result.get("bullets", [])
        # Keep bullets where at least some claims are supported
        kept    = [b for b in bullets if b.get("keep", True)]
        removed = [b for b in bullets if not b.get("keep", True)]
        final_bullets = kept
        verified_text = "\n".join(b.get("bullet_text", "") for b in kept if b.get("bullet_text",""))
    except Exception:
        verified_text  = draft
        final_bullets  = []

    # Step 3: Revise if verification removed bullets (fill gaps from evidence)
    if final_bullets and len(final_bullets) < len(shortlist):
        covered = {canon_recall_number(b.get("recall_number","")) for b in final_bullets}
        missing = [x for x in shortlist if canon_recall_number(x.get("recall_number","")) not in covered][:3]
        if missing:
            fill_prompt = (
                "The following recalls were not covered in the verified summary. "
                "Add one short, evidence-backed bullet for each, using the contrastive format "
                "'previously X, now Y' if old/new values are available.\n\n"
                "MISSING RECALLS:\n" + json.dumps([{
                    "recall_number": x["recall_number"],
                    "old_new_changes": x.get("old_new_pairs",[]),
                    "hazard_description": x.get("hazard_description",""),
                    "consumer_action": x.get("consumer_action",""),
                } for x in missing], ensure_ascii=False, indent=2)
                + "\n\nEVIDENCE:\n" + "\n\n---\n\n".join(ev_texts[:8])
            )
            fill_text, u3 = call_llm([
                {"role": "system", "content": "Be concise and evidence-grounded."},
                {"role": "user",   "content": fill_prompt},
            ])
            verified_text = (verified_text.strip() + "\n" + fill_text.strip()).strip()
            u2 = {k: u2.get(k,0) + v for k, v in u3.items() if isinstance(v, (int,float))}

    usage = {}
    for d in [u1, u2]:
        for k, v in d.items():
            if isinstance(v, (int,float)):
                usage[k] = usage.get(k,0) + v

    raw, final = _apply_enforce(verified_text, evidence, shortlist, max_unique=len(shortlist) or 10)

    return raw, final, evidence, {
        "mode":                "llm",
        "usage":               usage,
        "query_specs":         query_specs,
        "shortlist":           shortlist,
        "route_digest":        route_digest,
        "verification_result": verification_result,
        "structured_pairs":    structured_pairs,
    }


## 8) GOLD STANDARD

Gold standard: Read human gold frozen

In [ ]:
# 8) GOLD STANDARD


def load_human_gold(path: str) -> Optional[pd.DataFrame]:
    if not path or not os.path.exists(path):
        return None
    df = pd.read_csv(path, dtype=str)
    for c in df.columns:
        df[c] = df[c].map(normalize_text)
    return df

def build_proxy_gold(cu: pd.DataFrame, top_n: int = 10) -> Dict[str, Any]:
    """Proxy gold for development only. NOTE: uses importance_score ranking —
    do NOT use for final evaluation (circular with retrieval)."""
    if len(cu) == 0:
        return {"reference_recalls": [], "gold_change_units": [], "gold_mode": "proxy_debug"}
    top_recalls = (
        cu.sort_values("importance_score", ascending=False)
          .drop_duplicates(subset=[KEY_COL], keep="first")[KEY_COL]
          .head(top_n).astype(str).tolist()
    )
    gold_units = (
        cu[(cu["change_type"] != "modified") | (cu.get("changed_field","") != "")]
          .sort_values("importance_score", ascending=False)
          [["change_unit_id", KEY_COL, "change_type",
            cu.columns[cu.columns.str.lower() == "changed_field"][0] if any(cu.columns.str.lower() == "changed_field") else "change_type",
            "importance_score"]].head(20).to_dict(orient="records")
    )
    return {"reference_recalls": top_recalls, "gold_change_units": gold_units, "gold_mode": "proxy_debug"}

def build_gold_from_human(hdf: pd.DataFrame) -> Dict[str, Any]:
    if hdf is None or len(hdf) == 0:
        return {"reference_recalls": [], "gold_change_units": [], "gold_mode": "human_final"}

    df = hdf.copy()
    # Normalise column names
    rename = {"recall_number": KEY_COL, "field_name": "changed_field"}
    for a, b in rename.items():
        if a in df.columns and b not in df.columns:
            df[b] = df[a]

    # Filter to annotated-as-include rows
    for col in ["gold_include_in_summary", "is_real_change"]:
        if col in df.columns:
            df = df[df[col].str.lower().isin(["1","true","yes"])].copy()
            break

    def _imp_weight(x):
        s = str(x).lower()
        return 3 if s in ["high","5","4"] else (2 if s in ["medium","3","2"] else 1)

    df["importance_weight"] = df.get("importance", pd.Series(["low"] * len(df))).map(_imp_weight)

    ref_recalls = df[KEY_COL].dropna().astype(str).drop_duplicates().tolist()
    records = []
    for _, r in df.iterrows():
        records.append({
            "change_unit_id":   r.get("change_unit_id", f"{r.get(KEY_COL,'')}::{r.get('change_type','')}"),
            KEY_COL:            r.get(KEY_COL, ""),
            "change_type":      r.get("change_type", ""),
            "changed_field":    r.get("changed_field", ""),
            "importance_weight": int(r.get("importance_weight", 1)),
            "old_value":        r.get("old_value", ""),
            "new_value":        r.get("new_value", ""),
        })
    return {"reference_recalls": ref_recalls, "gold_change_units": records, "gold_mode": "human_final"}

def choose_gold(cu: pd.DataFrame, human_gold_df: Optional[pd.DataFrame]) -> Dict[str, Any]:
    if human_gold_df is not None and len(human_gold_df) > 0:
        return build_gold_from_human(human_gold_df)
    print("WARNING: Using proxy gold — not suitable for final evaluation.")
    return build_proxy_gold(cu)


## 9) EVALUATION

In [ ]:
# 9) EVALUATION


def extract_claims(text: str) -> List[str]:
    return [s for s in sent_split(normalize_text(text)) if len(s.split()) >= 4]

def recall_overlap_metrics(summary: str, ref_recalls: List[str]) -> Dict[str, Any]:
    cited = {canon_recall_number(x) for x in extract_recall_numbers(summary) if canon_recall_number(x)}
    ref   = {canon_recall_number(x) for x in ref_recalls if canon_recall_number(x)}
    if not ref:
        return {"recall_coverage": 0.0, "recall_precision": 0.0, "recall_f1": 0.0,
                "n_cited": len(cited), "n_reference": 0}
    cov  = len(cited & ref) / len(ref)
    prec = len(cited & ref) / len(cited) if cited else 0.0
    f1   = 2 * cov * prec / (cov + prec) if (cov + prec) > 0 else 0.0
    return {
        "recall_coverage":  round(cov,  4),
        "recall_precision": round(prec, 4),
        "recall_f1":        round(f1,   4),
        "n_cited":          len(cited),
        "n_reference":      len(ref),
    }

def retrieval_metrics(retrieved: pd.DataFrame, gold: Dict[str, Any]) -> Dict[str, Any]:
    ref = {canon_recall_number(x) for x in gold.get("reference_recalls",[]) if canon_recall_number(x)}
    if not ref or len(retrieved) == 0:
        return {f"retrieval_p@{k}": 0.0 for k in RETRIEVAL_KS} | \
               {f"retrieval_r@{k}": 0.0 for k in RETRIEVAL_KS}
    out = {}
    ret_recalls = [canon_recall_number(x) for x in
                   retrieved.get("recall_number", pd.Series(dtype=str)).astype(str).tolist()
                   if canon_recall_number(x)]
    for k in RETRIEVAL_KS:
        top_k = ret_recalls[:k]
        hits  = sum(1 for x in top_k if x in ref)
        out[f"retrieval_p@{k}"] = round(hits / k, 4)
        out[f"retrieval_r@{k}"] = round(hits / len(ref), 4)
    return out

def summary_change_unit_coverage(summary: str, gold: Dict[str, Any]) -> Dict[str, Any]:
    cited  = {canon_recall_number(x) for x in extract_recall_numbers(summary) if canon_recall_number(x)}
    units  = gold.get("gold_change_units", [])
    if not units:
        return {"cu_coverage": 0.0, "cu_weighted_coverage": 0.0, "n_gold_units": 0}

    total_weight, hit_weight, total_units, hit_units = 0, 0, 0, 0
    for u in units:
        rn = canon_recall_number(u.get(KEY_COL,""))
        w  = int(u.get("importance_weight", 1))
        total_weight += w; total_units += 1
        if rn and rn in cited:
            hit_weight += w; hit_units += 1

    return {
        "cu_coverage":          round(hit_units  / total_units,  4) if total_units  else 0.0,
        "cu_weighted_coverage": round(hit_weight / total_weight, 4) if total_weight else 0.0,
        "n_gold_units":         total_units,
    }

def claim_grounding_metrics_three_tier(summary: str,
                                       evidence_df: pd.DataFrame,
                                       verification_result: Optional[Dict] = None
                                       ) -> Dict[str, Any]:
    """
    Three-tier claim grounding: fully_supported / partially_supported / unsupported.

    If verification_result is available (from contrastive adaptive RAG self-verification),
    use it directly. Otherwise fall back to lexical overlap with two thresholds.
    """
    claims = extract_claims(summary)
    n      = len(claims)
    if n == 0:
        return {
            "claim_count": 0,
            "fully_supported_rate":    0.0,
            "partially_supported_rate": 0.0,
            "unsupported_rate":        0.0,
            "grounding_rate":          0.0,  # fully + partially
        }

    # Use structured verification if available
    if verification_result and "bullets" in verification_result:
        counts = Counter(b.get("overall_verdict","unsupported")
                         for b in verification_result["bullets"])
        total  = sum(counts.values()) or n
        return {
            "claim_count":              total,
            "fully_supported_rate":    round(counts.get("fully_supported",    0) / total, 4),
            "partially_supported_rate": round(counts.get("partially_supported", 0) / total, 4),
            "unsupported_rate":        round(counts.get("unsupported",         0) / total, 4),
            "grounding_rate":          round((counts.get("fully_supported",0) +
                                              counts.get("partially_supported",0)) / total, 4),
            "source": "self_verification",
        }

    # Lexical fallback — two thresholds for partial vs full
    ev_text = "\n".join(evidence_df["text"].fillna("").astype(str).tolist()
                        if len(evidence_df) else []).lower()
    stop = {"which","there","their","these","those","about","consumers",
            "product","recall","important","recall","changes"}

    fully, partially, unsupported = 0, 0, 0
    cited_recalls = {r for c in claims for r in extract_recall_numbers(c)}

    for claim in claims:
        lc   = claim.lower()
        crs  = extract_recall_numbers(claim)
        toks = [t for t in re.findall(r"[A-Za-z][A-Za-z\-]{4,}", lc) if t not in stop]

        recall_hit  = any(r.lower() in ev_text for r in crs) if crs else False
        if toks:
            overlap = sum(1 for t in toks[:10] if t in ev_text)
            ratio   = overlap / min(len(toks), 10)
        else:
            ratio = 0.0

        if recall_hit and ratio >= 0.5:
            fully += 1
        elif recall_hit or ratio >= 0.25:
            partially += 1
        else:
            unsupported += 1

    return {
        "claim_count":              n,
        "fully_supported_rate":    round(fully       / n, 4),
        "partially_supported_rate": round(partially   / n, 4),
        "unsupported_rate":        round(unsupported  / n, 4),
        "grounding_rate":          round((fully + partially) / n, 4),
        "source": "lexical",
    }

def priority_alignment_metrics(summary: str, cu: Optional[pd.DataFrame]) -> Dict[str, Any]:
    if cu is None or len(cu) == 0:
        return {"priority_alignment_top3": 0.0, "priority_alignment_top5": 0.0, "route_diversity": 0}
    cu_aug = augment_change_units(cu)
    cited  = {canon_recall_number(x) for x in extract_recall_numbers(summary) if canon_recall_number(x)}
    top_df = (cu_aug.sort_values(["route_priority","importance_score"], ascending=[False,False])
                    .drop_duplicates(subset=["canonical_recall_number"], keep="first"))
    top3 = set(top_df["canonical_recall_number"].head(3).tolist())
    top5 = set(top_df["canonical_recall_number"].head(5).tolist())
    routes_mentioned = set(
        cu_aug[cu_aug["canonical_recall_number"].isin(cited)]["route_label"].astype(str).tolist()
    )
    return {
        "priority_alignment_top3": round(len(cited & top3) / max(len(top3),1), 4),
        "priority_alignment_top5": round(len(cited & top5) / max(len(top5),1), 4),
        "route_diversity":         int(len(routes_mentioned)),
    }

def actionability_metrics(summary: str) -> Dict[str, Any]:
    action_terms = ["stop use","stop using","refund","repair","replace","return",
                    "immediately","take away","contact","dispose","unplug","keep away"]
    text_l  = normalize_text(summary).lower()
    claims  = extract_claims(summary)
    hits    = sum(1 for t in action_terms if t in text_l)
    act_claims = sum(1 for c in claims if any(t in c.lower() for t in action_terms))
    return {
        "actionability_term_hits":  hits,
        "actionable_claim_ratio":   round(act_claims / max(len(claims),1), 4),
    }

def route_coverage_metrics(summary: str, cu: Optional[pd.DataFrame]) -> Dict[str, Any]:
    if cu is None or len(cu) == 0:
        return {"route_coverage_rate": 0.0, "covered_routes": ""}
    cu_aug = augment_change_units(cu)
    cited  = {canon_recall_number(x) for x in extract_recall_numbers(summary) if canon_recall_number(x)}
    route_recalls = defaultdict(set)
    for _, r in cu_aug.drop_duplicates(subset=["canonical_recall_number"]).iterrows():
        rn = canon_recall_number(r.get("canonical_recall_number",""))
        if rn:
            route_recalls[normalize_text(r.get("route_label",""))].add(rn)
    hit_routes = [rt for rt, rns in route_recalls.items() if rns & cited]
    total_routes = sum(1 for rns in route_recalls.values() if rns)
    return {
        "route_coverage_rate": round(len(hit_routes) / max(total_routes,1), 4),
        "covered_routes":      " | ".join(sorted(hit_routes)),
    }

def auto_eval_one(method: str,
                  raw_summary: str,
                  final_summary: str,
                  gold: Dict[str, Any],
                  evidence_df: Optional[pd.DataFrame] = None,
                  cu: Optional[pd.DataFrame] = None,
                  verification_result: Optional[Dict] = None) -> Dict[str, Any]:
    """
    Evaluate both raw (pre-enforce) and final (post-enforce) summaries.
    Returns a single dict with raw_* and final_* prefixed metrics.
    """
    ev = evidence_df if evidence_df is not None else pd.DataFrame(columns=["recall_number","text"])
    ref_recalls = gold.get("reference_recalls", [])

    out = {"method": method, "gold_mode": gold.get("gold_mode","")}

    for prefix, text in [("raw", raw_summary), ("final", final_summary)]:
        out[f"{prefix}_summary_length_words"] = len(text.split())
        out.update({f"{prefix}_{k}": v for k, v in recall_overlap_metrics(text, ref_recalls).items()})
        out.update({f"{prefix}_{k}": v for k, v in summary_change_unit_coverage(text, gold).items()})
        # Use verification_result only for contrastive method's final summary
        vr = verification_result if (prefix == "final" and method == "contrastive_adaptive_rag") else None
        out.update({f"{prefix}_{k}": v for k, v in
                    claim_grounding_metrics_three_tier(text, ev, vr).items()})
        out.update({f"{prefix}_{k}": v for k, v in priority_alignment_metrics(text, cu).items()})
        out.update({f"{prefix}_{k}": v for k, v in actionability_metrics(text).items()})
        out.update({f"{prefix}_{k}": v for k, v in route_coverage_metrics(text, cu).items()})

    # Enforce delta — shows how much enforce_shortlist_coverage contributed
    out["enforce_delta_recall_coverage"] = round(
        out.get("final_recall_coverage", 0.0) - out.get("raw_recall_coverage", 0.0), 4
    )

    # Retrieval metrics only for RAG-based methods
    if method in ["rag", "agentic_rag", "contrastive_adaptive_rag"]:
        out.update(retrieval_metrics(ev, gold))

    return out


## 10) HUMAN EVAL TEMPLATES


In [ ]:
# 10) HUMAN EVAL TEMPLATES


def build_human_eval_templates(cu: pd.DataFrame,
                               outputs: Dict[str, Tuple[str, str]],
                               gold: Dict[str, Any],
                               output_dir: str = OUTPUT_DIR) -> Dict[str, str]:
    """
    outputs = {"method_name": (raw_summary, final_summary), ...}
    Exports annotation-ready CSVs. Includes IAA column for second annotator.
    """
    paths = {}

    # 1. Gold change units annotation sheet
    cu_aug = augment_change_units(cu)
    cu_export = cu_aug[[
        "change_unit_id", KEY_COL, "change_type",
        *[c for c in ["changed_field"] if c in cu_aug.columns],
        "importance_score", "route_label", "route_priority",
        *[c for c in ["old_value","new_value"] if c in cu_aug.columns],
    ]].head(50).copy()
    cu_export["gold_include_in_summary"] = ""   # annotator fills: 1/0
    cu_export["importance"]              = ""   # annotator fills: high/medium/low
    cu_export["annotator_1_label"]       = ""   # for IAA
    cu_export["annotator_2_label"]       = ""   # for IAA
    p = os.path.join(output_dir, "gold_change_units_for_annotation.csv")
    cu_export.to_csv(p, index=False)
    paths["gold_annotation"] = p

    # 2. Summary evaluation sheet (raw + final, both for annotation)
    rows = []
    for method, (raw, final) in outputs.items():
        for version, text in [("raw", raw), ("final", final)]:
            for bullet in [b.strip("- ").strip() for b in text.split("\n") if b.strip().startswith("-") or
                           re.match(r"^Recall\s+\d{2}-\d{3,4}", b.strip())]:
                if bullet:
                    rows.append({
                        "method": method, "version": version, "bullet": bullet,
                        "human_coverage_1to5": "",     # annotator fills
                        "human_accuracy_1to5": "",
                        "human_actionability_1to5": "",
                        "annotator_2_coverage": "",    # IAA second annotator
                        "notes": "",
                    })
    p = os.path.join(output_dir, "summary_eval_sheet.csv")
    pd.DataFrame(rows).to_csv(p, index=False)
    paths["summary_eval"] = p

    # 3. Claim grounding annotation sheet (three-tier)
    claim_rows = []
    for method, (_, final) in outputs.items():
        for claim in extract_claims(final):
            claim_rows.append({
                "method": method, "claim": claim,
                "verdict": "",              # annotator fills: fully_supported / partially_supported / unsupported
                "evidence_snippet": "",     # annotator pastes supporting text
                "annotator_2_verdict": "",  # IAA
            })
    p = os.path.join(output_dir, "claim_eval_sheet.csv")
    pd.DataFrame(claim_rows).to_csv(p, index=False)
    paths["claim_eval"] = p

    print(f"Human eval templates written to {output_dir}")
    return paths


## 11) PIPELINE

Overall process: Read snapshot, detect changes, build corpus/retriever, run four methods, evaluate, save results.

In [ ]:
# 11) PIPELINE


def run_pipeline(previous_snapshot_path: str,
                 current_snapshot_path: str,
                 output_dir: str = OUTPUT_DIR) -> Dict[str, Any]:
    os.makedirs(output_dir, exist_ok=True)
    check_required_previous_snapshot(previous_snapshot_path)

    prev_df = load_snapshot(previous_snapshot_path)
    cur_df  = load_snapshot(current_snapshot_path)
    print(f"Prev rows: {len(prev_df)} | Cur rows: {len(cur_df)}")

    cu           = augment_change_units(build_change_units(prev_df, cur_df))
    change_stats = summarize_change_stats(cu)
    print("Change stats:", change_stats)

    corpus    = build_retrieval_corpus(cu)
    retriever = SimpleRetriever(corpus)

    human_gold_df = load_human_gold(HUMAN_GOLD_PATH) if HUMAN_GOLD_PATH else None
    gold = choose_gold(cu, human_gold_df)
    print(f"Gold mode: {gold['gold_mode']} | Reference recalls: {len(gold['reference_recalls'])}")

    # ── Run all four methods ───────────────────
    print("Running prompt_only …")
    po_raw, po_final, po_meta = generate_prompt_only(cu, corpus)

    print("Running rag …")
    rag_raw, rag_final, rag_ev, rag_meta = generate_rag(cu, retriever)

    print("Running agentic_rag …")
    ag_raw, ag_final, ag_ev, ag_meta = generate_agentic_rag(cu, retriever)

    print("Running contrastive_adaptive_rag …")
    ca_raw, ca_final, ca_ev, ca_meta = generate_contrastive_adaptive_rag(cu, retriever)

    # ── Evaluate all four methods ──────────────
    auto_rows = [
        auto_eval_one("prompt_only",              po_raw, po_final, gold, corpus, cu),
        auto_eval_one("rag",                      rag_raw, rag_final, gold, rag_ev, cu),
        auto_eval_one("agentic_rag",              ag_raw,  ag_final,  gold, ag_ev,  cu),
        auto_eval_one("contrastive_adaptive_rag", ca_raw,  ca_final,  gold, ca_ev,  cu,
                      ca_meta.get("verification_result")),
    ]
    auto_eval_df = pd.DataFrame(auto_rows)

    # ── Save outputs ───────────────────────────
    summaries = {
        "prompt_only":              (po_raw, po_final),
        "rag":                      (rag_raw, rag_final),
        "agentic_rag":              (ag_raw,  ag_final),
        "contrastive_adaptive_rag": (ca_raw,  ca_final),
    }

    result = {
        "change_stats":    change_stats,
        "gold":            gold,
        "summaries":       {k: {"raw": v[0], "final": v[1]} for k, v in summaries.items()},
        "auto_eval_df":    auto_eval_df,
        "route_digest":    build_route_digest(cu),
        "meta": {
            "prompt_only":              po_meta,
            "rag":                      rag_meta,
            "agentic_rag":              ag_meta,
            "contrastive_adaptive_rag": ca_meta,
        },
        "evidence": {
            "rag":                      rag_ev,
            "agentic_rag":              ag_ev,
            "contrastive_adaptive_rag": ca_ev,
        },
    }

    # Write files
    write_json({k: {"raw": v[0], "final": v[1]} for k, v in summaries.items()},
               os.path.join(output_dir, "summaries.json"))
    auto_eval_df.to_csv(os.path.join(output_dir, "auto_eval.csv"), index=False)
    write_json(change_stats, os.path.join(output_dir, "change_stats.json"))
    write_json(build_route_digest(cu), os.path.join(output_dir, "route_digest.json"))

    # Human eval templates
    build_human_eval_templates(cu, summaries, gold, output_dir)

    print("\n── Auto Evaluation Results ──")
    cols = [c for c in auto_eval_df.columns if any(
        k in c for k in ["method","recall_coverage","recall_f1","cu_weighted",
                          "grounding_rate","enforce_delta","route_coverage"]
    )]
    print(auto_eval_df[cols].to_string(index=False))

    return result


## 12) ENTRY POINT

In [ ]:
# 12) ENTRY POINT


def run(previous_snapshot_path: str = EXPLICIT_PREVIOUS_SNAPSHOT) -> Dict[str, Any]:
    current_snapshot_path = fetch_current_snapshot(SNAPSHOT_DIR)
    return run_pipeline(previous_snapshot_path, current_snapshot_path, OUTPUT_DIR)


## 13) Pre-operation check


In [ ]:
# Path check before running the full pipeline
print("ROOT_DIR:", ROOT_DIR)
print("SNAPSHOT_DIR:", SNAPSHOT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPLICIT_PREVIOUS_SNAPSHOT:", EXPLICIT_PREVIOUS_SNAPSHOT)
print("Previous snapshot exists:", os.path.exists(EXPLICIT_PREVIOUS_SNAPSHOT))
print("HUMAN_GOLD_PATH:", HUMAN_GOLD_PATH if HUMAN_GOLD_PATH else "<empty: proxy gold mode>")
print("Human gold exists:", os.path.exists(HUMAN_GOLD_PATH) if HUMAN_GOLD_PATH else False)
print("LLM_MODE:", LLM_MODE)
print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))

if not os.path.exists(EXPLICIT_PREVIOUS_SNAPSHOT):
    print("\nACTION REQUIRED: upload the previous snapshot CSV or update EXPLICIT_PREVIOUS_SNAPSHOT.")
if not HUMAN_GOLD_PATH:
    print("\nNOTE: HUMAN_GOLD_PATH is empty. The run will use proxy_debug gold, suitable only for debugging.")
elif not os.path.exists(HUMAN_GOLD_PATH):
    print("\nWARNING: HUMAN_GOLD_PATH is set but the file does not exist.")


ROOT_DIR: /content/drive/MyDrive/GR5293 HW/final project/try
SNAPSHOT_DIR: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots
OUTPUT_DIR: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots
EXPLICIT_PREVIOUS_SNAPSHOT: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots/cpsc_recalls_20260304.csv
Previous snapshot exists: True
HUMAN_GOLD_PATH: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots/frozen_human_gold_current_window_v2.csv
Human gold exists: True
LLM_MODE: openai_compatible
OPENAI_API_KEY set: True


## Integrated Final Improvement Patch: Retrieval Coverage, Evidence Repair, and Fair Evaluation

The purpose of this patch is to make the four-method comparison more reliable and easier to evaluate. The original pipeline could miss important recall numbers because retrieval was sometimes too keyword-dependent, and the generated summaries did not always cover the strongest evidence even when that evidence had already been retrieved. This patch addresses those issues by improving route-balanced retrieval, expanding recall-number-based evidence search, repairing missing evidence coverage in RAG-style summaries, keeping Prompt-Only as a fair baseline, and improving the final Contrastive Adaptive RAG summary with verification and precision cleanup.


In [ ]:
# ============================================================
# Integrated Final Patch Loader
# Retrieval Coverage, Evidence Repair, and Final Method Stabilization
# ============================================================
#
# This loader executes the exact merged patch file.
# The patch file preserves the original V5, V5.1, V5.2, Fairness,
# Final, V6, and V7.1 patch logic and execution order.
#
# Why this is better than rewriting the patch:
# - It keeps the notebook clean.
# - It avoids changing the original patch behavior.
# - It reduces the risk of lower evaluation scores caused by refactoring.
#
# Run this cell after all base function definitions and before:
#     result = run(EXPLICIT_PREVIOUS_SNAPSHOT)
# ============================================================

import os

PATCH_PATH = "/content/drive/MyDrive/GR5293 HW/final project/try/snapshots/exact_merged_original_patches.py"

assert os.path.exists(PATCH_PATH), f"Patch file not found: {PATCH_PATH}"

with open(PATCH_PATH, "r", encoding="utf-8") as f:
    patch_code = f.read()

exec(compile(patch_code, PATCH_PATH, "exec"))

print("Exact merged final patch loaded.")
print("Next step: result = run(EXPLICIT_PREVIOUS_SNAPSHOT)")

V5 patch loaded
RAG_MAX_DOCS: 18
AGENT_MAX_DOCS: 22
RETRIEVAL_KS: [5, 10, 20]
V6 contrastive patch loaded
CONTRASTIVE_MAX_DOCS: 24
CONTRASTIVE_MAX_BULLETS: 16
V7.1 coverage-preserving precision cleanup loaded.
CONTRASTIVE_V71_MAX_FINAL_RECALLS: 22
Exact merged final patch loaded.
Next step: result = run(EXPLICIT_PREVIOUS_SNAPSHOT)


In [ ]:
# Run the full pipeline

result = run(EXPLICIT_PREVIOUS_SNAPSHOT)

# Show the main automatic evaluation table
from IPython.display import display
display(result["auto_eval_df"])


Using cached snapshot: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots/cpsc_recalls_20260430.csv
Previous snapshot OK: /content/drive/MyDrive/GR5293 HW/final project/try/snapshots/cpsc_recalls_20260304.csv
Prev rows: 9631 | Cur rows: 9722
Change stats: {'total': 412, 'added': 91, 'removed': 0, 'modified': 321, 'recalls_affected': 139}
Gold mode: human_final | Reference recalls: 16
Running prompt_only …
Running rag …
Running agentic_rag …
Running contrastive_adaptive_rag …
Generating final_meta_summary …
Human eval templates written to /content/drive/MyDrive/GR5293 HW/final project/try/snapshots

── Auto Evaluation Results ──
                  method  raw_recall_coverage  raw_recall_f1  raw_cu_weighted_coverage  raw_grounding_rate  raw_route_coverage_rate  final_recall_coverage  final_recall_f1  final_cu_weighted_coverage  final_grounding_rate  final_route_coverage_rate  enforce_delta_recall_coverage
             prompt_only               0.5000         0.6154              

,method,gold_mode,raw_summary_length_words,raw_recall_coverage,raw_recall_precision,raw_recall_f1,raw_n_cited,raw_n_reference,raw_cu_coverage,raw_cu_weighted_coverage,...,final_actionable_claim_ratio,final_route_coverage_rate,final_covered_routes,enforce_delta_recall_coverage,retrieval_p@5,retrieval_r@5,retrieval_p@10,retrieval_r@10,retrieval_p@20,retrieval_r@20
0,prompt_only,human_final,331,0.5000,0.8000,0.6154,10,16,0.5000,0.5000,...,0.0000,1.0,consumer_action_update | new_high_risk,0.0000,NaN,NaN,NaN,NaN,NaN,NaN
1,rag,human_final,460,0.6250,0.6250,0.6250,16,16,0.6250,0.6250,...,0.7895,1.0,consumer_action_update | new_high_risk,0.0625,1.0,0.3125,0.9,0.5625,0.55,0.6875
2,agentic_rag,human_final,309,0.7500,0.7500,0.7500,16,16,0.7500,0.7500,...,0.1562,1.0,consumer_action_update | new_high_risk,0.1250,1.0,0.3125,0.9,0.5625,0.70,0.8750
3,contrastive_adaptive_rag,human_final,670,0.6875,0.6875,0.6875,16,16,0.6875,0.6875,...,0.2545,1.0,consumer_action_update | new_high_risk,0.2500,1.0,0.3125,0.9,0.5625,0.75,0.9375


# Demo

In [ ]:

# 16) PRESENTATION DEMO CONTROL PANEL


# Options:
# "auto_best"
# "prompt_only"
# "rag"
# "agentic_rag"
# "contrastive_adaptive_rag"

DEMO_METHOD = "auto_best"

# Leave empty to automatically select one recall from the best summary.
# Or set, for example:
# DEMO_FOCUS_RECALL = "26-344"
DEMO_FOCUS_RECALL = ""

# Text preview length for tables
DEMO_TEXT_PREVIEW_LEN = 280

print("Demo control panel loaded.")
print("DEMO_METHOD:", DEMO_METHOD)
print("DEMO_FOCUS_RECALL:", DEMO_FOCUS_RECALL if DEMO_FOCUS_RECALL else "[auto]")

Demo control panel loaded.
DEMO_METHOD: auto_best
DEMO_FOCUS_RECALL: [auto]


In [ ]:
# 17) LIVE DEMO DASHBOARD


import os
import glob
import re
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 180)


def demo_warning(msg):
    display(Markdown(f"""
> **Demo Warning:** {msg}
"""))


def demo_short(x, n=280):
    try:
        x = normalize_text(str(x))
    except Exception:
        x = str(x)
    return x[:n] + ("..." if len(x) > n else "")


def demo_find_latest_snapshot(snapshot_dir):
    paths = glob.glob(os.path.join(snapshot_dir, "cpsc_recalls_*.csv"))
    if not paths:
        return None
    return max(paths, key=os.path.getmtime)


def demo_extract_recalls(text):
    try:
        nums = extract_recall_numbers(text or "")
        out = []
        seen = set()
        for x in nums:
            rn = canon_recall_number(x)
            if rn and rn not in seen:
                out.append(rn)
                seen.add(rn)
        return out
    except Exception:
        return re.findall(r"\b\d{2}-\d{3}\b", str(text or ""))


def demo_select_best_method(eval_df):
    df = eval_df.copy()

    ranking_cols = [
        "final_recall_f1",
        "final_recall_coverage",
        "final_cu_weighted_coverage",
        "final_grounding_rate",
    ]

    for c in ranking_cols:
        if c not in df.columns:
            df[c] = 0.0

    best_row = (
        df.sort_values(ranking_cols, ascending=False)
          .iloc[0]
    )

    return best_row["method"], best_row


def demo_get_summary(result, method):
    try:
        return result["summaries"][method]["final"]
    except Exception:
        return ""


def demo_get_evidence(result, method):
    try:
        if method == "prompt_only":
            return pd.DataFrame()
        return result.get("evidence", {}).get(method, pd.DataFrame())
    except Exception:
        return pd.DataFrame()


def demo_rebuild_change_units():
    try:
        current_snapshot_path = demo_find_latest_snapshot(SNAPSHOT_DIR)
        if current_snapshot_path is None:
            return None, None, None, None

        prev_df = load_snapshot(EXPLICIT_PREVIOUS_SNAPSHOT)
        cur_df = load_snapshot(current_snapshot_path)
        cu = augment_change_units(build_change_units(prev_df, cur_df))
        corpus = build_retrieval_corpus(cu)

        return current_snapshot_path, prev_df, cur_df, cu

    except Exception as e:
        demo_warning(f"Could not rebuild change units. Error: {e}")
        return None, None, None, None


def demo_filter_recall_df(df, recall_number):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()

    candidate_cols = ["recall_number", KEY_COL, "Recall Number"]
    recall_col = None

    for c in candidate_cols:
        if c in out.columns:
            recall_col = c
            break

    if recall_col is None:
        return pd.DataFrame()

    out["_canon_recall"] = out[recall_col].astype(str).map(canon_recall_number)
    return out[out["_canon_recall"] == recall_number].copy()


def demo_choose_focus_recall(summary, gold_recalls, cu):
    # 1. Use user-specified recall if available
    if DEMO_FOCUS_RECALL:
        rn = canon_recall_number(DEMO_FOCUS_RECALL)
        if rn:
            return rn

    # 2. Prefer a recall appearing in both generated summary and gold set
    summary_recalls = demo_extract_recalls(summary)
    gold_set = set([canon_recall_number(x) for x in gold_recalls])

    for rn in summary_recalls:
        if rn in gold_set:
            return rn

    # 3. Otherwise use the first recall mentioned in summary
    if summary_recalls:
        return summary_recalls[0]

    # 4. Otherwise use the first change unit
    if cu is not None and len(cu) > 0 and KEY_COL in cu.columns:
        return canon_recall_number(cu.iloc[0][KEY_COL])

    return ""


def demo_summary_for_recall(summary, recall_number):
    if not summary or not recall_number:
        return ""

    lines = []
    for line in str(summary).splitlines():
        if recall_number in line:
            lines.append(line.strip())

    if lines:
        return "\n".join(lines)

    pattern = rf"(Recall\s+{re.escape(recall_number)}\:.*?)(?=\n\s*[-*]?\s*Recall\s+\d{{2}}-\d{{3}}\:|$)"
    m = re.search(pattern, summary, flags=re.S)

    if m:
        return m.group(1).strip()

    return ""



# 0. Validate result


if "result" not in globals():
    demo_warning("`result` is not found. Please run `result = run(EXPLICIT_PREVIOUS_SNAPSHOT)` first.")
else:
    display(Markdown("# Live Demo: CPSC Recall Change Monitoring System"))


    # 1. Select method


    eval_df = result.get("auto_eval_df", pd.DataFrame())

    if eval_df is None or len(eval_df) == 0:
        demo_warning("No automatic evaluation table found in result.")
    else:
        if DEMO_METHOD == "auto_best":
            selected_method, best_row = demo_select_best_method(eval_df)
        else:
            selected_method = DEMO_METHOD
            matched = eval_df[eval_df["method"] == selected_method]
            best_row = matched.iloc[0] if len(matched) > 0 else pd.Series(dtype=object)

        selected_summary = demo_get_summary(result, selected_method)

        if not selected_summary:
            demo_warning(f"No final summary found for method `{selected_method}`.")


        # 2. Rebuild demo data


        current_snapshot_path, prev_df, cur_df, cu = demo_rebuild_change_units()

        gold_recalls = result.get("gold", {}).get("reference_recalls", [])
        focus_recall = demo_choose_focus_recall(selected_summary, gold_recalls, cu)


        # 3. Demo overview


        display(Markdown(f"""
## 1. Demo Setup

This demo shows the complete workflow from CPSC recall snapshots to an evaluated risk summary.

| Item | Value |
|---|---|
| Previous snapshot | `{os.path.basename(EXPLICIT_PREVIOUS_SNAPSHOT)}` |
| Current snapshot | `{os.path.basename(current_snapshot_path) if current_snapshot_path else "N/A"}` |
| Previous rows | `{len(prev_df) if prev_df is not None else "N/A"}` |
| Current rows | `{len(cur_df) if cur_df is not None else "N/A"}` |
| Detected change units | `{len(cu) if cu is not None else "N/A"}` |
| Demo method | `{selected_method}` |
| Demo focus recall | `{focus_recall}` |

**Live interaction option:** during the presentation, I can change `DEMO_METHOD` or `DEMO_FOCUS_RECALL` and rerun this dashboard.
"""))


        # 4. Step 1: Change detection


        display(Markdown("## 2. Step 1 — Change Detection"))

        if cu is None or len(cu) == 0:
            demo_warning("No change units were detected.")
        else:
            cu_focus = demo_filter_recall_df(cu, focus_recall)

            change_cols = [
                KEY_COL,
                "change_type",
                "changed_field",
                "old_value",
                "new_value",
                "importance_score",
                "route_label",
            ]

            change_cols = [c for c in change_cols if c in cu_focus.columns]

            if len(cu_focus) == 0:
                demo_warning(f"No detailed change-unit row found for recall `{focus_recall}`.")
            else:
                show_change = cu_focus[change_cols].copy()

                for c in ["old_value", "new_value"]:
                    if c in show_change.columns:
                        show_change[c] = show_change[c].map(
                            lambda x: demo_short(x, DEMO_TEXT_PREVIEW_LEN)
                        )

                display(Markdown("""
The system first compares the previous and current snapshots and converts differences into structured change units.
"""))
                display(show_change.head(5))


        # 5. Step 2: Evidence retrieval


        display(Markdown("## 3. Step 2 — Evidence Retrieval"))

        ev_df = demo_get_evidence(result, selected_method)

        if ev_df is None or len(ev_df) == 0:
            demo_warning(
                f"No explicit evidence table found for `{selected_method}`. "
                "This is expected for prompt-only, because prompt-only does not use retrieval."
            )
        else:
            ev_focus = demo_filter_recall_df(ev_df, focus_recall)

            evidence_cols = [
                "recall_number",
                KEY_COL,
                "change_types",
                "changed_fields",
                "route_label",
                "summary_score",
                "hybrid_score",
                "balanced_score",
                "text",
            ]

            evidence_cols = [c for c in evidence_cols if c in ev_focus.columns]

            if len(ev_focus) == 0:
                demo_warning(f"No retrieved evidence row found for recall `{focus_recall}`.")
            else:
                show_ev = ev_focus[evidence_cols].copy()

                if "text" in show_ev.columns:
                    show_ev["text"] = show_ev["text"].map(
                        lambda x: demo_short(x, DEMO_TEXT_PREVIEW_LEN)
                    )

                display(Markdown("""
The RAG-based methods retrieve evidence before generation, which makes the output more auditable than pure prompt-only summarization.
"""))
                display(show_ev.head(3))


        # 6. Step 3: Generated summary


        display(Markdown("## 4. Step 3 — Generated Summary for the Focus Recall"))

        focus_text = demo_summary_for_recall(selected_summary, focus_recall)

        if focus_text:
            display(Markdown(focus_text))
        else:
            demo_warning(f"The selected summary does not contain a direct bullet for `{focus_recall}`.")


        # 7. Step 4: Method comparison


        display(Markdown("## 5. Step 4 — Baseline Comparison and Evaluation"))

        eval_cols = [
            "method",
            "gold_mode",
            "final_recall_coverage",
            "final_recall_precision",
            "final_recall_f1",
            "final_cu_coverage",
            "final_cu_weighted_coverage",
            "final_grounding_rate",
            "final_route_coverage",
            "retrieval_p@5",
            "retrieval_r@5",
            "retrieval_p@10",
            "retrieval_r@10",
        ]

        eval_cols = [c for c in eval_cols if c in eval_df.columns]

        sort_cols = [
            c for c in [
                "final_recall_f1",
                "final_recall_coverage",
                "final_cu_weighted_coverage",
                "final_grounding_rate",
            ]
            if c in eval_df.columns
        ]

        if sort_cols:
            display(
                eval_df[eval_cols]
                .sort_values(sort_cols, ascending=False)
                .reset_index(drop=True)
            )
        else:
            display(eval_df[eval_cols])


        # 8. Step 5: Technical depth


        display(Markdown(f"""
## 6. Technical Depth Explanation

**Core technical contributions shown in this demo:**

1. **Structured change detection**
   The system does not summarize raw CSV rows directly. It first converts snapshot differences into structured change units.

2. **Retrieval-augmented generation**
   RAG and agentic RAG retrieve supporting evidence before generation, reducing unsupported claims.

3. **Contrastive adaptive RAG**
   The strongest method uses retrieval coverage, route labels, and evidence repair to improve recall coverage while preserving grounding.

4. **Evaluation-driven method selection**
   The final output is selected using automatic metrics rather than manually choosing the most convincing-looking summary.

5. **Scalability design**
   The same workflow can be scheduled daily or weekly: fetch snapshot → detect changes → generate summary → evaluate → export report.

**Selected method for this demo:** `{selected_method}`
"""))

# Live Demo: CPSC Recall Change Monitoring System


## 1. Demo Setup

This demo shows the complete workflow from CPSC recall snapshots to an evaluated risk summary.

| Item | Value |
|---|---|
| Previous snapshot | `cpsc_recalls_20260304.csv` |
| Current snapshot | `cpsc_recalls_20260430.csv` |
| Previous rows | `9631` |
| Current rows | `9722` |
| Detected change units | `412` |
| Demo method | `contrastive_adaptive_rag` |
| Demo focus recall | `26-418` |

**Live interaction option:** during the presentation, I can change `DEMO_METHOD` or `DEMO_FOCUS_RECALL` and rerun this dashboard.


## 2. Step 1 — Change Detection


The system first compares the previous and current snapshots and converts differences into structured change units.


,Recall Number,change_type,changed_field,old_value,new_value,importance_score,route_label
0,26-418,added,,,,49.5,new_high_risk


...

## 4. Step 3 — Generated Summary for the Focus Recall

- Recall 26-418: Casely Wireless Portable Power Banks — newly added recall. Hazard: The recalled lithium-ion battery can overheat and ignite, posing a risk of serious injury or death from fire and burn hazards. Action required: Consumers should immediately stop using the recalled power banks and contact Casely for a free replacement.

## 5. Step 4 — Baseline Comparison and Evaluation

,method,gold_mode,final_recall_coverage,final_recall_precision,final_recall_f1,final_cu_coverage,final_cu_weighted_coverage,final_grounding_rate,retrieval_p@5,retrieval_r@5,retrieval_p@10,retrieval_r@10
0,contrastive_adaptive_rag,human_final,0.9375,0.6522,0.7692,0.9375,0.9375,1.0,1.0,0.3125,0.9,0.5625
1,agentic_rag,human_final,0.8750,0.6087,0.7179,0.8750,0.8750,1.0,1.0,0.3125,0.9,0.5625
2,rag,human_final,0.6875,0.5789,0.6286,0.6875,0.6875,1.0,1.0,0.3125,0.9,0.5625
3,prompt_only,human_final,0.5000,0.8000,0.6154,0.5000,0.5000,1.0,NaN,NaN,NaN,NaN



## 6. Technical Depth Explanation

**Core technical contributions shown in this demo:**

1. **Structured change detection**  
   The system does not summarize raw CSV rows directly. It first converts snapshot differences into structured change units.

2. **Retrieval-augmented generation**  
   RAG and agentic RAG retrieve supporting evidence before generation, reducing unsupported claims.

3. **Contrastive adaptive RAG**  
   The strongest method uses retrieval coverage, route labels, and evidence repair to improve recall coverage while preserving grounding.

4. **Evaluation-driven method selection**  
   The final output is selected using automatic metrics rather than manually choosing the most convincing-looking summary.

5. **Scalability design**  
   The same workflow can be scheduled daily or weekly: fetch snapshot → detect changes → generate summary → evaluate → export report.

**Selected method for this demo:** `contrastive_adaptive_rag`


In [ ]:
# 18) SELECT BEST METHOD AND EXPORT FINAL SUMMARY TEXT FILE


import os
import pandas as pd
from IPython.display import display, Markdown

if "result" not in globals():
    raise RuntimeError("Please run `result = run(EXPLICIT_PREVIOUS_SNAPSHOT)` first.")

eval_df = result["auto_eval_df"].copy()

ranking_cols = [
    "final_recall_f1",
    "final_recall_coverage",
    "final_cu_weighted_coverage",
    "final_grounding_rate",
]

for c in ranking_cols:
    if c not in eval_df.columns:
        eval_df[c] = 0.0

best_row = (
    eval_df.sort_values(ranking_cols, ascending=False)
    .iloc[0]
)

best_method = best_row["method"]
best_summary_text = result["summaries"][best_method]["final"]

display(Markdown(f"""
# Final Exported Summary

**Best method selected by evaluation:** `{best_method}`

The system selects the best method based on:

1. final recall F1
2. final recall coverage
3. weighted change-unit coverage
4. grounding rate

"""))

display(
    eval_df[
        [
            "method",
            "final_recall_coverage",
            "final_recall_precision",
            "final_recall_f1",
            "final_cu_weighted_coverage",
            "final_grounding_rate",
        ]
    ].sort_values(ranking_cols, ascending=False)
)

display(Markdown("## Best Method Final Summary"))
display(Markdown(best_summary_text))


summary_txt_path = os.path.join(
    OUTPUT_DIR,
    f"best_final_summary_{best_method}.txt"
)

with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write("CPSC Recall Monitoring Final Summary\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Best Method: {best_method}\n\n")
    f.write("Evaluation Scores:\n")
    f.write(f"- final_recall_coverage: {best_row.get('final_recall_coverage', 0):.4f}\n")
    f.write(f"- final_recall_precision: {best_row.get('final_recall_precision', 0):.4f}\n")
    f.write(f"- final_recall_f1: {best_row.get('final_recall_f1', 0):.4f}\n")
    f.write(f"- final_cu_weighted_coverage: {best_row.get('final_cu_weighted_coverage', 0):.4f}\n")
    f.write(f"- final_grounding_rate: {best_row.get('final_grounding_rate', 0):.4f}\n\n")
    f.write("=" * 80 + "\n\n")
    f.write("Final Summary:\n\n")
    f.write(best_summary_text)

print("Final summary text file saved to:")
print(summary_txt_path)


# Final Exported Summary

**Best method selected by evaluation:** `contrastive_adaptive_rag`

The system selects the best method based on:

1. final recall F1  
2. final recall coverage  
3. weighted change-unit coverage  
4. grounding rate  



,method,final_recall_coverage,final_recall_precision,final_recall_f1,final_cu_weighted_coverage,final_grounding_rate
3,contrastive_adaptive_rag,0.9375,0.6522,0.7692,0.9375,1.0
2,agentic_rag,0.8750,0.6087,0.7179,0.8750,1.0
1,rag,0.6875,0.5789,0.6286,0.6875,1.0
0,prompt_only,0.5000,0.8000,0.6154,0.5000,1.0


## Best Method Final Summary

Contrastive adaptive recall watchlist:
- Recall 26-418: Casely Wireless Portable Power Banks — newly added recall. Hazard: The recalled lithium-ion battery can overheat and ignite, posing a risk of serious injury or death from fire and burn hazards. Action required: Consumers should immediately stop using the recalled power banks and contact Casely for a free replacement.
- Recall 26-401: Male-to-Male Extension Cords — newly added recall. Hazard: The exposed prongs can become energized, posing a risk of serious injury and death from electrocution. Action required: Consumers should stop using the recalled extension cords immediately and contact Shenzhen Shijingjie Network Technology for a full refund.
- Recall 26-379: HTRC and Haisito Battery Chargers — newly added recall. Hazard: The chargers can ignite or cause a connected battery to ignite, posing a fire hazard. Action required: Consumers should immediately stop using the recalled chargers and contact Huizhou Haitan Technology for instructions on returning the chargers for a full refund.
- Recall 26-331: BUILT LUUM Tumblers — newly added recall. Hazard: The LED tumblers can break, making button cell batteries accessible to children, posing choking and ingestion hazards. Action required: Consumers should stop using the recalled tumblers immediately and contact Lifetime Brands for a full refund.
- Recall 26-344: Sunnyyes LED Mini Lights — newly added recall. Hazard: The lights contain lithium coin batteries that can be accessed easily by children, posing an ingestion hazard. Action required: Consumers should stop using the recalled LED lights immediately and contact Sunnyyes for a full refund.
- Recall 26-398: ShymeryDirect LED Lights — newly added recall. Hazard: The lights contain lithium coin batteries that can be accessed easily by children, posing an ingestion hazard. Action required: Consumers should immediately stop using the recalled LED lights and contact Shymery for a full refund.
- Recall 26-403: Happiness Light LED Lights — newly added recall. Hazard: The lights contain lithium coin batteries that can be accessed easily by children, posing an ingestion hazard. Action required: Consumers should stop using the recalled LED lights immediately and contact Happiness Light for a full refund.
- Recall 26-348: FUNTOK 24V 2-Seater Ride-On Trucks — newly added recall. Hazard: The truck’s circuit board can overheat and ignite, posing fire and burn hazards. Action required: Consumers should stop using the recalled ride-on truck immediately and contact Shenzhen Luobei Trading Co. for a full refund.
- Recall 26-434: Autobrush Sonic Pro Kids Toothbrush Boxes — newly added recall. Hazard: The boxes contain a lithium coin battery that can be accessed by children, posing an ingestion hazard. Action required: Consumers should stop using the boxes immediately and contact Autobrush for a $5 refund.
- Recall 26-424: LED Finger Beam Lights — newly added recall. Hazard: The toys contain button cell batteries that can be accessed easily by children, posing an ingestion hazard. Action required: Consumers should stop using the LED Finger Beam Lights immediately and contact ZMC Group for a full refund.
- Recall 26-428: Lil' Buddies Pet Laser Toy — newly added recall. Hazard: The battery compartment is not secure, making button cell batteries easily accessible to children, posing an ingestion hazard. Action required: Consumers should stop using the recalled pet toys and contact JC Sales for a full refund.
- Recall 18-040: Change: modified. Fields: Description | Remedy.
- Recall 25-157: Change: modified. Fields: Remedy | Remedy Type.
- Recall 09-195: Change: modified. Fields: Description | Hazard Description | Importers | Incidents | Manufactured In | Name of product | Remedy | Remedy Type | Units.
- Recall 08-565: Change: modified. Fields: Description | Hazard Description | Importers | Incidents | Manufactured In | Name of product | Remedy | Remedy Type | Units. Additional evidence-backed recall changes:
- Recall 08-570: Change: modified. Fields: Description | Hazard Description | Incidents | Manufactured In | Name of product | Remedy | Remedy Type | Units. Previously Remedy: ; now: Consumers should stop using these recalled ATVs immediately and contact any Honda ATV dealer to make an appointment for a free repair. Registered owners of the .
- Recall 26-389: Silks Recall Children’s Loungewear Sets Due to Risk of Serious Injury or Death from Burn Hazard; Violates Mandatory Flammability Standards for Children’s Sleepwear. Hazard: The recalled children’s loungewear violates mandatory flammability standards for children’s sleepwear, posing a risk of serious injuries or deadly burn hazards to children. Action required: Consumers should immediately stop using the recalled loungewear, take it away from children and contact Silks for a full refund or store credit.
- Recall 26-346: Change: added.
- Recall 26-396: Change: added.
- Recall 26-369: Change: added.
- Recall 26-317: Change: added.
- Recall 26-291: Change: modified. Fields: Hazard Description. Previously Hazard Description: The magnetic stick figure toy sets violate&nbsp;https://www.ecfr.gov/current/title-16/chapter-II/subchapter-B/part-1250… mandatory standard for toys because the; now: The magnetic stick figure toy sets violate&nbsp;https://www.ecfr.gov/current/title-16/chapter-II/subchapter-B/part-1250… mandatory standard for toys because the.

Final summary text file saved to:
/content/drive/MyDrive/GR5293 HW/final project/try/snapshots/best_final_summary_contrastive_adaptive_rag.txt
